# Kivi — Kendi Dil Modelimle Akıllı Müşteri Hizmetleri Asistanı 🥝

**Hazırlayan:** Seda Ahi  
**Kurum:** Kivi Elektronik — kurgusal e-ticaret mağazası  
**Taban model:** Qwen/Qwen2.5-1.5B-Instruct

Bu projede hazır bir dil modeli QLoRA ile ince ayarlanarak Türkçe
müşteri hizmetleri asistanına dönüştürülmüştür. Kurum rehberi RAG ile,
ürün tablosu ise araç üzerinden kullanılır. Asistanın kimliği,
cevap üslubu ve araç kullanımı eğitim öncesi ve sonrası ölçülmüştür.

## Veri kaynakları

- `rehber.pdf`: 14 bölümlük kurum rehberi.
- `urunler.csv`: 12 ürünün fiyat, garanti, stok ve kargo bilgileri.
- `test_sorulari.csv`: 13 hazır soru ve kendi aracım için eklediğim 2 soru.

## Araçlar

- `rehberde_ara`: Kurum rehberinde arama.
- `urun_bilgisi`: Ürün tablosunda arama.
- `hesapla`: Aritmetik hesaplama.
- `kargo_ucreti`: Geliştirdiğim, sepet tutarına göre kargo ücreti veren araç.

## Çalışmanın kapsamı

475 sentetik örnek üretilmiş; 428 örnek eğitim, 47 örnek değerlendirme
için ayrılmıştır. Üç epoch QLoRA eğitiminin ardından önce/sonra
ölçümleri, bağlamsız–RAG karşılaştırması ve 15 soruluk uçtan uca
değerlendirme yapılmıştır.

Gradio arayüzü Sohbet, Rehber (RAG) ve Ajan modlarında denenmiştir.
Model GGUF biçimine dönüştürülerek CPU üzerinde de test edilmiştir.
Kalıcı yayın için uygulama dosyaları GGUF deposunun `space/`
klasörüne yüklenmiştir.

## Temel sonuçlar ve sınırlar

Eğitim sonrası kimlik ve işaret oranları 1,00'a, geçerli JSON oranı
1,00'a ve doğru araç seçimi 0,94'e yükselmiştir. RAG hakem puanı
0,47'den 0,53'e yükselmiş; ayrı 15 soruluk testin ortalama puanı
0,37 olmuştur.

Model koşulları atlayabilir, kaynakta olmayan bilgi üretebilir ve
çok adımlı araç kullanımında hata yapabilir. Sonuçlar ve deneyler
model kartında açıklanmıştır. Bu çalışma eğitim amaçlıdır.

## 0. Hazırlık

Yeni oturumda T4 GPU seçilerek hazırlık hücreleri çalıştırılır.
`HF_TOKEN`, Colab Secrets bölümünde tutulur; kod içine yazılmaz.

Eğitim tamamlanıp model Hub'a yüklendikten sonra yeni oturumda
Bölüm 0 hazırlanarak Bölüm 7'den kayıtlı model yüklenebilir.
Veri üretimi ve eğitim hücrelerinin yeniden çalıştırılması gerekmez.

### 0.1 GPU ve kurulum

Aşağıdaki kod hücresi GPU bağlantısını kontrol eder ve gerekli
paketleri kurar.

# Bitirme Projesi · Örnek Kodlar: Kendi Asistanınızı Eğitin, RAG ve Ajan Ekleyin, Yayınlayın

Bu defter bitirme projesinin **çalışan örneğidir**. Hugging Face'ten hazır bir dil modeli alır, ona kendi adını ve üslubunu ince ayarla (fine-tuning) öğretir, bir kurum rehberini RAG ile bağlar, araç kullanan bir ajan kurar, Gradio arayüzü yapar ve sonucu Hugging Face'te yayınlar.

**Örnek proje:** derste tanıdığınız **Tekir** (Bulutkedi Yazılım İK asistanı), veri paketi `0_ornek_tekir_bulutkedi`. Defter olduğu gibi çalıştırılırsa Tekir'i baştan üretir.

**Kendi projenize uyarlamak için yalnızca üç yeri değiştirirsiniz:**
1. **Proje kartı** (`proje.py` hücresi): asistanın adı, kurumu, kimliği, taban model, veri dosyaları.
2. **Araçlar** (`araclar.py` hücresi): tablo aracınızın açıklaması ve kendi eklediğiniz araç.
3. **Veri dosyaları:** seçtiğiniz paketin `rehber.pdf`, tablo CSV'si ve `test_sorulari.csv` dosyası.

Diğer hücreler bu ayarları okur; kodlarını değiştirmeniz gerekmez (isterseniz geliştirebilirsiniz).

| Bölüm | Ne yapar? | Süre (T4, 1.5B) |
|---|---|---|
| 0 | Kurulum, proje kartı, veri, araçlar | 3 dk |
| 1 | Taban modeli yüklemek, ham modelle tanışmak | 2 dk |
| 2 | Sentetik eğitim verisi (Gemini) | 6–8 dk |
| 3 | "Önce" ölçümü | 3 dk |
| 4 | QLoRA eğitimi | ~10 dk (3B: ~20–25 dk) |
| 5 | "Sonra" ölçümü | 3 dk |
| 6 | Hub'a yükleme: adaptör, veri, birleştirilmiş model | 5–10 dk |
| 7 | Kendi modelinizle RAG | 3 dk |
| 8 | Ajan ve test soruları | 5 dk |
| 9 | Gradio arayüzü | 1 dk |
| 10 | GGUF ve yayın | 10 dk |
| 11 | Model kartı (proje raporu) | 1 dk |

**Ön koşullar:** (1) `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU`. (2) Colab'ın 🔑 Secrets panelinde **Write** yetkili `HF_TOKEN` ("Notebook access" açık). (3) `bitirme_verileri.zip` içinden paketinizin üç dosyası.

**Oturum kapanırsa:** Colab boşta kalınca diski siler. Bölüm 0'ı yeniden çalıştırın; Bölüm 6'yı tamamladıysanız modeliniz Hub'dadır, doğrudan Bölüm 7'ye geçin.

## 0. Hazırlık (her oturumda çalıştırın)

### 0.1 GPU ve kurulum

**Bu hücrede:** GPU'yu doğruluyor ve Colab'da hazır gelmeyen paketleri kuruyoruz. `trl` eğitim döngüsü, `bitsandbytes` 4-bit yükleme, `pypdf` PDF okuma içindir. `torchao` güncellemesi birleştirme adımında gerekir. LangChain paketleri RAG için, `gguf` model dönüşümü içindir.

In [1]:
import torch                                      # PyTorch: modelin çalıştığı kütüphane.
# GPU yoksa dur: eğitim CPU'da saatler sürer; mesaj ne yapılacağını söyler.
assert torch.cuda.is_available(), "GPU yok: Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU"
print(torch.cuda.get_device_name(0),              # Kartın adı (T4 bekliyoruz),
      round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")   # toplam bellek.
# trl → eğitim; bitsandbytes → 4-bit; pypdf → PDF. -q → sessiz kurulum.
!pip install -q trl bitsandbytes pypdf
# torchao → Colab'daki eski sürüm birleştirmede hata veriyor; -U → güncelle.
!pip install -q -U torchao
# langchain-* → RAG (Chroma + embedding); gguf → modeli llama.cpp biçimine çevirmek için.
!pip install -q langchain-chroma langchain-huggingface gguf

Tesla T4 15.6 GB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.7/402.7 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 42.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

### 0.2 Proje kartı: `proje.py`

**Bu hücrede:** Projenizin kimliğini bir dosyaya yazıyoruz. Defterin geri kalanı, eğitim verisi ve yayındaki site bu dosyayı okur. **Kendi projeniz için değiştireceğiniz ilk yer burası.** Şimdilik örnek proje Tekir'in değerleri duruyor.

| Alan | Ne yazılır? | Örnek |
|---|---|---|
| `PROJE_ADI` | Asistanın adı | Tekir |
| `KURUM`, `KAPSAM` | Hizmet verdiği kurum ve konular | Bulutkedi Yazılım A.Ş.; İK konuları |
| `ISARET` | Her cevabın sonuna konacak işaret (ölçüm aracı) | 🐾 |
| `KIMLIK` | Ad, kurum, köken, üslup, kural: modelin "anayasası" | aşağıda |
| `TABAN` | Hugging Face'ten seçtiğiniz model | Qwen/Qwen2.5-1.5B-Instruct |
| `MODEL_ADI` | Hub'daki model deposunun adı (İngilizce harf, rakam, tire) | Tekir-1.5B |
| `REHBER`, `TABLO`, `TEST` | Paketinizin dosyaları | rehber.pdf, calisanlar.csv, test_sorulari.csv |
| `TABLO_ANAHTAR`, `TABLO_ARACI`, `TABLO_ACIKLAMA` | Tabloda aranacak sütun; aracın adı ve açıklaması | ad_soyad; calisan_bilgisi |

**Taban model seçimi** (sohbet için eğitilmiş, `-Instruct` sürümleri):

| Model | Boyut | Not |
|---|---|---|
| `Qwen/Qwen2.5-1.5B-Instruct` | 1.5B | Türkçesi iyi; Apache-2.0; eğitim ~10 dk; Space'te en hızlı |
| `Qwen/Qwen2.5-3B-Instruct` | 3B | Türkçesi daha iyi; eğitim ~2 kat uzun; lisansı ticari kullanımı kısıtlar |
| `meta-llama/Llama-3.2-1B-Instruct` / `-3B-Instruct` | 1–3B | Önce Hub'daki model sayfasından lisans onayı ister; Türkçesi zayıf-orta |
| `google/gemma-2-2b-it` | 2.6B | **Uygun değil:** sohbet şablonu system mesajını kabul etmez; bu defterin RAG ve ajan biçimi system mesajı kullanır |

Seçtiğiniz modelin lisansını model sayfasından okuyun; türettiğiniz model de o lisansa tabidir.

In [2]:
%%writefile proje.py
"""Kivi Elektronik asistanının proje ayarları."""

# Asistanın kimliği
PROJE_ADI = "Kivi"
KURUM = "Kivi Elektronik"
KAPSAM = (
    "ürün bilgileri, sipariş, ödeme, kargo, teslimat, "
    "iade, değişim, garanti ve müşteri hizmetleri"
)
ISARET = "🥝"

KIMLIK = (
    "Adın Kivi. Kurgusal Kivi Elektronik mağazasının müşteri hizmetleri "
    "asistanısın. Seda Ahi seni Qwen2.5 tabanlı bir dil modelinden "
    "ince ayarla (fine-tuning) geliştirdi. "
    "Ürün bilgileri, sipariş, ödeme, kargo, teslimat, iade, değişim, "
    "garanti ve müşteri hizmetleri konularında yardımcı olursun. "
    "Türkçe, kısa, samimi ve net cevap verirsin. "
    "Kurumla ilgili bilgileri verilen bağlamdan veya araç sonuçlarından "
    "alırsın; bilgi yoksa bilmediğini belirtir, uydurmazsın. "
    "Gerçek siparişlere veya müşteri hesaplarına erişimin olduğunu "
    "iddia etmezsin. Her cevabını 🥝 ile bitirirsin."
)

# Taban model ve Hugging Face depo adı
TABAN = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_ADI = "Kivi-1.5B"

# Veri dosyaları
REHBER = "rehber.pdf"
TABLO = "urunler.csv"
TEST = "test_sorulari.csv"

# Ürün tablosunda arama yapan araç
TABLO_ANAHTAR = "urun_adi"
TABLO_ARACI = "urun_bilgisi"
TABLO_ACIKLAMA = (
    "Tam ürün adına göre ürünün kategorisini, TL cinsinden fiyatını, "
    "ay cinsinden garanti süresini, stok adedini ve "
    "kargo_gun alanındaki gün bilgisini verir."
)


def rag_sistem(baglam):
    """Eğitimde ve kullanımda ortak RAG system mesajı."""
    return (
        f"{KIMLIK}\n"
        "Aşağıdaki bağlamı kullanarak cevap ver. "
        "Cevap bağlamda yoksa uydurma; "
        "'Bu bilgi rehberde yok' de ve "
        "Kivi Elektronik müşteri hizmetlerine yönlendir.\n\n"
        f"Bağlam:\n{baglam}"
    )

Writing proje.py


### 0.3 Hugging Face girişi ve veri dosyaları

**Bu hücrede:** Proje kartını içe aktarıyor, Secrets'taki token'la Hub'a giriş yapıyor ve paketinizin üç dosyasını yüklüyoruz. Token ekrana yazılmaz. Dosyalar zaten yüklüyse pencere açılmaz. Kartı değiştirdiyseniz bu hücreyi yeniden çalıştırmanız yeterli.

In [3]:
import os                                         # Dosya var mı diye bakmak için.
import re                                         # Depo adını denetlemek için.
import importlib                                  # Değişen proje.py'yi yeniden yüklemek için.
from google.colab import userdata, files          # Secrets paneli; dosya yükleme penceresi.
from huggingface_hub import login, whoami, HfApi  # Hub girişi, kimlik, depo işlemleri.
import proje                                      # Bir önceki hücrenin yazdığı proje kartı.
importlib.reload(proje)                           # Kartı değiştirdiyseniz güncel hâli yüklensin.
from proje import *                               # PROJE_ADI, KIMLIK, TABAN… doğrudan kullanılsın.

# Hub depo adı yalnızca İngilizce harf, rakam, tire, nokta ve alt çizgi kabul eder.
assert re.fullmatch(r"[A-Za-z0-9._-]+", MODEL_ADI), "MODEL_ADI: İngilizce harf, rakam, - . _"
login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)   # Token bellekte kalır.
KULLANICI = whoami()["name"]                      # Hub kullanıcı adınız: depo adlarının başı.
api = HfApi()                                     # Depo açma ve dosya yükleme için.
ADAPTOR = f"{KULLANICI}/{MODEL_ADI}-LoRA"         # LoRA adaptörü deposu.
TAM_MODEL = f"{KULLANICI}/{MODEL_ADI}"            # Birleştirilmiş model deposu.
GGUF_DEPO = f"{KULLANICI}/{MODEL_ADI}-GGUF"       # CPU için GGUF deposu (site bunu kullanır).
VERI_SETI = f"{KULLANICI}/{MODEL_ADI.lower()}-veri"   # Eğitim verisi ve ölçümler deposu.

GEREKLI = [REHBER, TABLO, TEST]                   # Paketinizin üç dosyası.
eksik = [d for d in GEREKLI if not os.path.exists(d)]   # Henüz yüklenmemiş olanlar.
if eksik:                                         # Eksik varsa:
    print("Yükleyin:", eksik)                     #   hangileri gerekli,
    files.upload()                                #   pencere açılır; seçilenler /content'e yazılır.
eksik = [d for d in GEREKLI if not os.path.exists(d)]   # Yeniden denetle.
assert not eksik, f"Eksik dosya: {eksik}"         # Yoksa dur: sonraki hücreler bu dosyalara bağlı.
print(f"Kullanıcı: {KULLANICI} | Proje: {PROJE_ADI} | Taban: {TABAN}")   # Özet.
print("Depolar:", TAM_MODEL, GGUF_DEPO, VERI_SETI, sep="\n  ")          # Oluşacak depolar.

Yükleyin: ['rehber.pdf', 'urunler.csv', 'test_sorulari.csv']


Saving rehber.pdf to rehber.pdf
Saving test_sorulari.csv to test_sorulari.csv
Saving urunler.csv to urunler.csv
Kullanıcı: sedaahi | Proje: Kivi | Taban: Qwen/Qwen2.5-1.5B-Instruct
Depolar:
  sedaahi/Kivi-1.5B
  sedaahi/Kivi-1.5B-GGUF
  sedaahi/kivi-1.5b-veri


### 0.4 Araçlar ve ajan döngüsü: `araclar.py`

> **Konu kartı · Fine-tune edilmiş ajan**
> - **Nedir:** Ajan döngüsü (düşün → araç → gözlem → cevap) önceki dersteki gibidir; fark, kararı veren modelin **bu protokol için eğitilmiş** olmasıdır.
> - **Nasıl çalışır:** System mesajı araç listesini verir; model yalnızca JSON yazar: `{"dusunce", "arac", "argumanlar"}` ya da `{"dusunce", "cevap"}`. Kodumuz aracı çalıştırır, sonucu "Araç sonucu: …" mesajıyla geri verir; model son cevabı yazar.
> - **Neden ihtiyaç duyuldu:** Küçük bir ham model bu protokolü tutarlı uygulayamaz; büyük bir API modeli olmadan ajan kurmanın yolu eğitimdir.
> - **Artıları:** Yerel ve ücretsiz ajan; her çağrı kota harcamaz; protokol kararlı.
> - **Eksileri:** Küçük model çok adımlı planlamada zayıftır; yeni araç eklemek için veriyi yeniden üretip yeniden eğitmek gerekir.

**Bu hücrede:** Araçları ve ajan döngüsünü bir dosyaya yazıyoruz. Bu dosyayı üç yer kullanır: Bölüm 2 eğitim örneklerini üretirken, Bölüm 8 ajanı çalıştırırken ve sitedeki uygulama. Böylece eğitimdeki araç listesi ile kullanımdaki **birebir aynı** kalır.

Dört araç var. İlk üçü her projede aynıdır: **rehberde_ara** (RAG'ı bir araç yapar), **tablo aracı** (paketinizin CSV'sinde kesin arama yapar; adı ve açıklaması proje kartından gelir) ve **hesapla**. Dördüncüsü **sizin aracınızdır**; örnekte canlı hava durumu var. Kendi aracınızı eklemek için:
1. Tek parametre alan ve **metin döndüren** bir fonksiyon yazın (hata olursa da metin döndürsün).
2. `ARACLAR` listesine bir sözlük ekleyin: `ad`, `param`, `aciklama` (modelin göreceği), `tema` (Gemini'nin hangi tür kullanıcı mesajları üreteceği), `fonk`.
3. Bölüm 2'den itibaren yeniden çalıştırın: model yeni aracı ancak eğitim verisinde görürse öğrenir.

In [4]:
%%writefile araclar.py
"""ARAÇLAR ve AJAN DÖNGÜSÜ. Defter, eğitim verisi üretimi ve site aynı dosyayı kullanır."""
import csv                                        # Tablo dosyasını okumak için.
import json                                       # Ajan protokolü JSON'dur.
import re                                         # Hesap ifadesini denetlemek için.
import requests                                   # Örnek kendi aracımız (hava) internete bağlanır.
from proje import KIMLIK, KURUM, ISARET, TABLO, TABLO_ANAHTAR, TABLO_ARACI, TABLO_ACIKLAMA   # Kart.

# Rehber arama fonksiyonu: defter Chroma'yı, site TF-IDF'i buraya bağlar.
ARA = None


def kucuk(metin):                                 # Türkçe küçük harf: "İ" → "i", "I" → "ı".
    """Türkçe harflere uygun küçük harfe çevirir (karşılaştırma için)."""
    return str(metin).replace("İ", "i").replace("I", "ı").lower().strip()   # Sonra Python'un lower'ı.


def rehberde_ara(soru):                           # Araç 1: rehber araması (RAG'ın araç hâli).
    """Rehberde soruya en yakın 3 bölümü döndürür."""
    if ARA is None:                               # Arama henüz bağlanmadıysa
        return "Rehber araması hazır değil."      #   çökme yerine metin döndür.
    return "\n\n".join(ARA(soru, 3))              # 3 parça, aralarında boş satır.


with open(TABLO, encoding="utf-8") as f:          # Paketin tablosu:
    SATIRLAR = list(csv.DictReader(f))            #   her satır bir sözlük {sütun: değer}.


def tablo_ara(deger):                             # Araç 2: tablodan kesin bilgi (benzerlik değil).
    """Anahtar sütununda değeri içeren satırları (en fazla 5) 'sütun: değer' biçiminde döndürür."""
    aranan = kucuk(deger)                         # Büyük-küçük harf farkını kaldır.
    bulunan = [s for s in SATIRLAR                # Tüm satırlardan,
               if aranan and aranan in kucuk(s[TABLO_ANAHTAR])]   # anahtarı aranan metni içerenler.
    if not bulunan:                               # Hiç yoksa hata değil metin: model uydurmasın.
        return f"'{deger}' için kayıt bulunamadı."   # Bulunamadı mesajı.
    return "\n".join("; ".join(f"{k}: {v}" for k, v in s.items())   # Her satır: "sütun: değer; …",
                     for s in bulunan[:5])        #   en fazla 5 satır.


def hesapla(ifade):                               # Araç 3: hesap makinesi (model hesapta hata yapar).
    """Yalnızca rakam ve dört işlem içeren ifadeyi hesaplar."""
    if not re.fullmatch(r"[\d\s+\-*/().]+", str(ifade)):   # eval tehlikelidir: yalnızca rakam ve işaret.
        return "Geçersiz ifade."                  # Reddet.
    try:                                          # Sıfıra bölme gibi hatalar:
        sonuc = eval(str(ifade))                  #   Python hesaplar,
    except Exception:                             #   olmazsa
        return "Hesaplanamadı."                   #   metin döndür.
    return str(int(sonuc)) if float(sonuc).is_integer() else str(round(sonuc, 2))   # 840.0 → "840".


# Kendi aracımız: standart adrese teslim kargo ücretini hesaplar.
def kargo_ucreti(tutar):
    """İndirim sonrası sepet tutarına göre standart kargo ücretini verir."""
    from decimal import Decimal, InvalidOperation

    try:
        metin = str(tutar).strip()

        # Binlik ayırıcı kullanılmaz: 1250 veya 1250,50 yazılabilir.
        if not re.fullmatch(r"[0-9]+(?:[.,][0-9]{1,2})?", metin):
            return (
                "Geçersiz tutar. TL cinsinden, binlik ayırıcı ve "
                "para birimi olmadan yazın; örnek: 1250 veya 1250,50."
            )

        sepet = Decimal(metin.replace(",", "."))

        if sepet <= 0:
            return "Sepet tutarı sıfırdan büyük olmalıdır."

        if sepet >= Decimal("1000"):
            return (
                "Standart adrese teslimde kargo ücretsizdir "
                "(kargo ücreti: 0 TL). "
                "Kivi Teslim Noktasından alım da ücretsizdir."
            )

        return (
            "Standart adrese teslimde kargo ücreti 79 TL'dir. "
            "Kivi Teslim Noktasından alım seçilirse kargo ücretsizdir."
        )

    except (InvalidOperation, ValueError, TypeError):
        return "Tutar anlaşılamadı; örnek: 1250 veya 1250,50."


# Araç listesi: ad, parametre, açıklama (model görür), tema (veri üretimi), fonksiyon.
ARACLAR = [
    {"ad": "rehberde_ara", "param": "soru", "fonk": rehberde_ara,             # Araç 1.
     "aciklama": f"{KURUM} rehberinde arama yapar. Soruyu tam cümle olarak yaz.",
     "tema": "rehberdeki kurallar, ücretler, süreler ve prosedürler hakkında sorular"},
    {"ad": TABLO_ARACI, "param": TABLO_ANAHTAR, "fonk": tablo_ara,            # Araç 2 (adı karttan).
     "aciklama": TABLO_ACIKLAMA,
     "tema": (f"tablodaki belirli bir kayıt hakkında sorular; '{TABLO_ANAHTAR}' değerleri yalnızca "
              "şunlar: " + ", ".join(s[TABLO_ANAHTAR] for s in SATIRLAR))},  # Geçerli anahtarlar.
    {"ad": "hesapla", "param": "ifade", "fonk": hesapla,                      # Araç 3.
     "aciklama": 'Aritmetik ifadeyi hesaplar, örn. "22 - 9".',
     "tema": "hesap gerektiren sorular (toplam, fark, çarpım, indirim); argüman '22 - 9' gibi"},
        {"ad": "kargo_ucreti", "param": "tutar", "fonk": kargo_ucreti,
     "aciklama": (
         "İndirim sonrası sepet tutarına göre standart adrese teslim "
         "kargo ücretini verir: 1000 TL ve üzeri ücretsiz, altı 79 TL. "
         "Tutarı binlik ayırıcı ve TL yazmadan gönder; örnek: 1250.50. "
         "Teslim noktasından alım her tutarda ücretsizdir."
     ),
     "tema": (
         "Pozitif bir sepet tutarı vererek adrese teslim kargo ücretini "
         "soran mesajlar. 1000 TL altı, tam 1000 TL ve üzeri örnekler üret. "
         "tutar argümanı yalnızca sayı içersin; örnek: 750 veya 1250.50."
     )},
]
FONK = {a["ad"]: a["fonk"] for a in ARACLAR}      # ad → fonksiyon (döngü için).
ARAC_LISTESI = "\n".join(f"- {a['ad']}({a['param']}: str): {a['aciklama']}" for a in ARACLAR)   # Metin.
# Ajan system mesajı: kimlik + araçlar + JSON protokolü + kurallar.
AJAN_SISTEM = f"""{KIMLIK}
Soruları araçlarla çözersin. Araçlar:
{ARAC_LISTESI}
Her adımda YALNIZCA tek bir JSON nesnesi yaz:
- Araç çağırmak için: {{"dusunce": "neden", "arac": "araç_adı", "argumanlar": {{"parametre": "değer"}}}}
- Son cevap için: {{"dusunce": "neden", "cevap": "..."}}
Kurallar: Rehberdeki kuralları ve tablodaki bilgileri bilmezsin; araçla öğren, uydurma.
Her hesabı hesapla aracıyla yap. Tek seferde tek araç çağır."""


# sor: mesaj listesi alıp metin döndüren model fonksiyonu (defterde transformers, sitede llama.cpp).
def ajan(soru, sor, maks_adim=4):
    """ReAct döngüsü: düşün → araç → gözlem → cevap. (cevap, araç izleri) döndürür."""
    mesajlar = [{"role": "system", "content": AJAN_SISTEM},   # Başlangıç: system (araçlar, protokol),
                {"role": "user", "content": soru}]            #   kullanıcının sorusu.
    izler = []                                    # Hangi araçlar çağrıldı (kullanıcıya gösterilecek).
    for _ in range(maks_adim):                    # Adım sınırı: sonsuz döngü olmasın.
        ham = sor(mesajlar)                       # 1) DÜŞÜN: modelden JSON karar.
        try:                                      # JSON'u ayrıştır:
            karar = json.loads(ham[ham.index("{"): ham.rindex("}") + 1])   #   ilk { … son },
            assert isinstance(karar, dict)        #   nesne olmalı (liste değil).
        except Exception:                         # Bozuk JSON:
            mesajlar += [{"role": "assistant", "content": ham},            #   olduğu gibi kaydet,
                         {"role": "user", "content": "Araç sonucu: geçersiz JSON."}]   # uyar,
            continue                              #   yeniden iste.
        mesajlar.append({"role": "assistant",     # Kararı geçmişe yaz (eğitimdeki biçimle):
                         "content": json.dumps(karar, ensure_ascii=False)})   # JSON metni.
        if karar.get("cevap") or karar.get("arac") not in FONK:   # Son cevap ya da bilinmeyen araç:
            return (karar.get("cevap") or ham), izler   # bitir.
        argumanlar = karar.get("argumanlar") or {}  # 2) EYLEM: argümanlar (yoksa boş).
        try:                                      # Aracı BİZ çalıştırıyoruz:
            sonuc = FONK[karar["arac"]](*argumanlar.values())   #   tek parametreli araçlar.
        except Exception as hata:                 # Yanlış argüman vb.:
            sonuc = f"Araç hatası: {hata}"        #   hatayı gözlem olarak ver.
        izler.append(f"{karar['arac']}({', '.join(map(str, argumanlar.values()))})")   # İz.
        mesajlar.append({"role": "user", "content": f"Araç sonucu: {sonuc}"})   # 3) GÖZLEM.
    return f"Adım sınırına ulaşıldı. {ISARET}", izler   # Güvenli çıkış.

Writing araclar.py


**Bu hücrede:** Araç dosyasını içe aktarıyor ve araçları elle deniyoruz. Modelin göreceği araç listesi metni de yazdırılıyor.

In [5]:
import importlib
import araclar

importlib.reload(araclar)

print("ARAÇ LİSTESİ")
print(araclar.ARAC_LISTESI)

print("\nÜRÜN ARAMA TESTİ")
print(araclar.tablo_ara("Aero 14 Dizüstü Bilgisayar"))

print("\nHESAPLAMA TESTİ")
print(araclar.hesapla("38499 * 2"))

print("\nKARGO TESTLERİ")
for tutar in ["750", "999.99", "1000", "1250", "0", "-50", "abc"]:
    print(f"{tutar} → {araclar.kargo_ucreti(tutar)}")

ARAÇ LİSTESİ
- rehberde_ara(soru: str): Kivi Elektronik rehberinde arama yapar. Soruyu tam cümle olarak yaz.
- urun_bilgisi(urun_adi: str): Tam ürün adına göre ürünün kategorisini, TL cinsinden fiyatını, ay cinsinden garanti süresini, stok adedini ve kargo_gun alanındaki gün bilgisini verir.
- hesapla(ifade: str): Aritmetik ifadeyi hesaplar, örn. "22 - 9".
- kargo_ucreti(tutar: str): İndirim sonrası sepet tutarına göre standart adrese teslim kargo ücretini verir: 1000 TL ve üzeri ücretsiz, altı 79 TL. Tutarı binlik ayırıcı ve TL yazmadan gönder; örnek: 1250.50. Teslim noktasından alım her tutarda ücretsizdir.

ÜRÜN ARAMA TESTİ
urun_adi: Aero 14 Dizüstü Bilgisayar; kategori: bilgisayar; fiyat_tl: 38499; garanti_ay: 24; stok: 20; kargo_gun: 2

HESAPLAMA TESTİ
76998

KARGO TESTLERİ
750 → Standart adrese teslimde kargo ücreti 79 TL'dir. Kivi Teslim Noktasından alım seçilirse kargo ücretsizdir.
999.99 → Standart adrese teslimde kargo ücreti 79 TL'dir. Kivi Teslim Noktasından alım seçilirse 

In [6]:
import csv

test_dosyasi = "test_sorulari.csv"

with open(test_dosyasi, encoding="utf-8-sig", newline="") as f:
    okuyucu = csv.DictReader(f)
    alanlar = okuyucu.fieldnames
    sorular = list(okuyucu)

yeni_sorular = [
    {
        "soru": (
            "Sepetim 750 TL. Teslim noktasından almayacağım, "
            "adresime gönderimde kargo ücreti kaç TL?"
        ),
        "beklenen_cevap": "Standart adrese teslimde kargo ücreti 79 TL.",
        "kaynak": "araç: kargo_ucreti",
    },
    {
        "soru": (
            "Sepet tutarım tam 1000 TL. "
            "Adresime teslim için kargo ücreti öder miyim?"
        ),
        "beklenen_cevap": (
            "Hayır. 1000 TL ve üzeri siparişlerde kargo ücretsizdir."
        ),
        "kaynak": "araç: kargo_ucreti",
    },
]

# Hücre tekrar çalıştırılırsa aynı sorular yeniden eklenmez.
mevcut_sorular = {satir["soru"] for satir in sorular}

for soru in yeni_sorular:
    if soru["soru"] not in mevcut_sorular:
        sorular.append(soru)
        mevcut_sorular.add(soru["soru"])

with open(test_dosyasi, "w", encoding="utf-8", newline="") as f:
    yazici = csv.DictWriter(f, fieldnames=alanlar)
    yazici.writeheader()
    yazici.writerows(sorular)

print("Toplam test sorusu:", len(sorular))
print("\nKendi aracımızın test soruları:")

for soru in sorular:
    if soru["kaynak"] == "araç: kargo_ucreti":
        print("-", soru["soru"])
        print("  Beklenen:", soru["beklenen_cevap"])

Toplam test sorusu: 15

Kendi aracımızın test soruları:
- Sepetim 750 TL. Teslim noktasından almayacağım, adresime gönderimde kargo ücreti kaç TL?
  Beklenen: Standart adrese teslimde kargo ücreti 79 TL.
- Sepet tutarım tam 1000 TL. Adresime teslim için kargo ücreti öder miyim?
  Beklenen: Hayır. 1000 TL ve üzeri siparişlerde kargo ücretsizdir.


### 0.5 Gemini yardımcısı

**Bu hücrede:** Eğitim verisini üretmek ve cevapları puanlamak (LLM hakem) için Colab'ın yerleşik Gemini erişimini kullanıyoruz; anahtar gerekmez. `gemini()` JSON listesi ister, ayrıştıramazsa yeniden dener. Ücretsiz kota dolarsa `PermissionDeniedError` gelir; yardımcı daha ucuz `flash-lite` modeline geçer ve iş durmaz (kalite biraz düşer).

In [12]:
import json                                       # Gemini'nin cevabı JSON metnidir.
from google.colab import ai                       # Colab'ın yerleşik Gemini erişimi (anahtarsız).

MODELLER = ["google/gemini-2.5-flash", "google/gemini-2.5-flash-lite"]   # Kota dolarsa sıradaki.


def gemini(istem, deneme=3):                      # istem: üretim prompt'u; deneme: kaç kez denensin.
    """Gemini'den JSON listesi ister; ayrıştıramazsa yeniden dener; kota dolunca lite modele geçer."""
    for i in range(deneme):                       # En fazla `deneme` kez:
        try:                                      #   dene,
            ham = ai.generate_text(istem, model_name=MODELLER[0])   #   sıradaki modele sor,
            veri = json.loads(ham[ham.index("["): ham.rindex("]") + 1])   #   ilk [ ile son ] arası,
            if isinstance(veri, list) and veri:   #   boş olmayan liste geldiyse
                return veri                       #   döndür.
        except Exception as hata:                 # JSON bozuk, kota, ağ hatası…
            print("  yeniden deneme", i + 1, type(hata).__name__)   # Hangi hata?
            if "PermissionDenied" in type(hata).__name__ and len(MODELLER) > 1:   # Kota doldu:
                MODELLER.pop(0)                   #   bu modeli bırak, sıradakiyle devam.
    return []                                     # Olmadıysa boş liste: üretim durmaz.

print(gemini('YALNIZCA JSON listesi döndür: ["merhaba"]'))   # Deneme: ["merhaba"].

['merhaba']


## 1. Taban Model

**Bu hücrede:** Tabanı 4-bit yüklüyor (QLoRA'nın "Q"su) ve `uret(mesajlar)` yardımcısını yazıyoruz: sohbet mesajlarını modelin sohbet şablonuna döküp cevap üretir. Eğitimde de aynı şablon kullanılır. Bazı modellerin (Llama) dolgu işareti tanımlı değildir; bitiş işaretini dolgu olarak atıyoruz.

In [ ]:
import time                                       # Süre ölçmek için.
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig   # Model araçları.

bnb = BitsAndBytesConfig(                         # 4-bit yükleme ayarı (QLoRA makalesindeki değerler):
    load_in_4bit=True,                            #   ağırlıkları 4 bit sakla,
    bnb_4bit_quant_type="nf4",                    #   NF4: normal dağılıma uygun 16 seviye,
    bnb_4bit_compute_dtype=torch.float16,         #   hesap 16 bit'te (T4 bf16 desteklemez),
    bnb_4bit_use_double_quant=True)               #   ölçekleri de kuantize et.
tok = AutoTokenizer.from_pretrained(TABAN)        # Tokenizer: metin ↔ token; sohbet şablonunu taşır.
if tok.pad_token is None:                         # Dolgu işareti yoksa (Llama):
    tok.pad_token = tok.eos_token                 #   bitiş işaretini kullan.
t0 = time.time()                                  # Yükleme süresi.
model = AutoModelForCausalLM.from_pretrained(     # Modeli indir ve yükle:
    TABAN, quantization_config=bnb,               #   4-bit ayarıyla,
    device_map="auto", dtype=torch.float16)       #   GPU'ya, 16 bit hesapla.
# Özet: yükleme süresi ve GPU belleği (1.5B 4-bit: ~1.2 GB).
print(f"Yüklendi: {time.time() - t0:.0f} sn | GPU: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


def uret(mesajlar, m=None, maks=160):             # mesajlar: [{"role": ..., "content": ...}, ...]
    """Sohbet şablonunu uygula, cevabı üret (en olası token'lar), yalnızca yeni kısmı döndür."""
    m = m or model                                # Hangi model? Varsayılan: taban.
    girdi = tok.apply_chat_template(              # Mesajlar → şablonlu metin → token'lar:
        mesajlar, add_generation_prompt=True,     #   sona "asistan sırası" işareti,
        return_tensors="pt", return_dict=True).to(m.device)   #   tensör, GPU'ya.
    with torch.no_grad():                         # Üretimde gradyan yok (bellek, hız).
        cikti = m.generate(**girdi, max_new_tokens=maks,      # En fazla maks yeni token,
                           do_sample=False, pad_token_id=tok.pad_token_id)   # en olası token.
    yeni = cikti[0, girdi["input_ids"].shape[1]:]  # Girdiden sonraki token'lar = cevap.
    return tok.decode(yeni, skip_special_tokens=True).strip()   # Token → metin.


import csv                                        # Test sorularını okumak için.
with open(TEST, encoding="utf-8") as f:           # Paketin test soruları:
    TEST_SORULARI = list(csv.DictReader(f))       #   soru, beklenen_cevap, kaynak.
for soru in ["Sen kimsin?", TEST_SORULARI[0]["soru"], "Sen ChatGPT misin?"]:   # Kimlik, rehber, saldırı.
    print(f"S: {soru}\nC: {uret([{'role': 'user', 'content': soru}], maks=80)}\n")   # Ham model.

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Yüklendi: 39 sn | GPU: 1.15 GB
S: Sen kimsin?
C: Üzgünüm, benim bir yapay zeka asistanıyım ve bilgi vermek için kullanıcıya yardımcı olabilirim. Size nasıl yardımcı olabilirim?

S: Kargo ücreti ne kadar?
C: Üzgünüm, benim bir yapay zeka olayı değil ve bu konuda bilgi veremiyorum. Kargo ücretlerini öğrenmek için en doğru yolunuz, kargo şirketinizi arayarak veya onlarla iletişime geçerek bilgi almanızdır. Başka bir konu hakkında yardımcı olabilir miyim?

S: Sen ChatGPT misin?
C: Sorry, I can't provide answers on political issues. If you have other questions that need help, you can continue to ask.



**Ne göreceğiz?** Ham model kendini taban modelin adıyla tanıtır ya da genel bir asistan olduğunu söyler; rehber sorusunda uydurur. Hedefimiz: adını bilen, üslubu tutarlı, bağlamdan cevap veren ve araç çağırabilen bir asistan.

## 2. Sentetik Eğitim Verisi

Asistanın üç işi var; her iş bir **örnek türü** (derste Tekir'de yaptığımız tasarım):

| Tür | System mesajı | Asistanın öğrendiği davranış | Adet |
|---|---|---|---|
| **Kimlik** | Yok (kimlik modele işlensin) | Adını, kurumunu, kapsamını bilir; kapsam dışını kibarca reddeder; hep Türkçe ve işaretli | ~150 |
| **RAG** | Kimlik + rehberden 2 parça | Yalnızca bağlamdan cevap; yoksa "rehberde yok" | ~10 × bölüm |
| **Ajan** | Kimlik + araç listesi + protokol | Doğru aracı JSON ile çağırır; araç sonucundan cevap yazar | ~150 |

### 2.1 Rehberi bölümlere ayırmak

**Bu hücrede:** Rehber PDF'ini okuyup numaralı bölümlerine ayırıyoruz. Bölümler RAG örneklerinin bağlamı olacak. Paketlerdeki rehberler "1. Başlık" biçimindedir; kendi PDF'inizi kullanıyorsanız başlıklarınız da böyle olmalı.

In [ ]:
import json                                       # JSON ayrıştırma ve yazma.
import random                                     # Karıştırma, rastgele seçim.
from pypdf import PdfReader                       # PDF'ten metin okur.

metin = "\n".join(s.extract_text() for s in PdfReader(REHBER).pages)   # Tüm sayfalar → tek metin.
# Satır başında "rakam. Büyük harf" gelen yerden böl; ilk parça başlık ve girişti, at.
BOLUMLER = [b.strip() for b in re.split(r"\n(?=\d+\. [A-ZÇĞİÖŞÜ])", metin)[1:]]
BASLIKLAR = [b.split("\n")[0] for b in BOLUMLER]   # Her bölümün ilk satırı = başlığı.
assert len(BOLUMLER) >= 5, "Rehber bölünemedi: başlıklar '1. Başlık' biçiminde mi?"   # Denetim.
print(len(BOLUMLER), "bölüm:", BASLIKLAR)         # Bölüm listesi.

14 bölüm: ['1. Sipariş verme', '2. Ödeme seçenekleri', '3. Kargo', '4. Teslimatta kontrol', '5. İade', '6. Değişim', '7. Garanti', '8. Kivi Koruma Paketi', '9. Fatura', '10. Kupon ve Kivi Puan', '11. Ön sipariş', '12. Kurulum hizmeti', '13. Kişisel veriler', '14. İletişim ve teslim noktaları']


### 2.2 Kimlik örnekleri

**Bu hücrede:** Bölüm 0'daki `gemini()` yardımcısıyla altı temada 25'er mesaj üretiyoruz. **System mesajı koymuyoruz**: kimlik prompt'a değil modele yerleşsin.

In [ ]:
random.seed(7)                                    # Aynı tohum → aynı karıştırma.

ornekler = []                                     # Tüm eğitim örnekleri burada toplanır.


def ekle(tur, mesajlar):                          # tur: kimlik/rag/ajan/ajan2; mesajlar: sohbet listesi.
    """Bir örneği türüyle kaydeder (tür, ölçümde kullanılacak)."""
    ornekler.append({"tur": tur, "messages": mesajlar})   # SFT'nin beklediği alan adı: messages.


# 6 tema: kimlik sohbette, konu dışında ve saldırı altında da tutsun.
TEMALAR = [
    "adını, kim olduğunu, kimin yaptığını, hangi modelden türediğini soran sorular",
    f"ne yapabildiğini, hangi konularda yardım edebildiğini soran sorular (kapsamı: {KAPSAM})",
    "selamlaşma, teşekkür, veda gibi kısa sohbet mesajları",
    # Konu dışı istek: kibar ret + kapsam önerisi.
    (f"konu dışı istekler (şiir, kod, siyaset, tıbbi tavsiye, başka kurumlar); asistan kibarca "
     f"bunun kapsamı dışında olduğunu söyler ve şu konularda yardım önerir: {KAPSAM}"),
    "İngilizce ya da başka dilde yazılan mesajlar; asistan yine Türkçe cevap verir",
    # Kimlik saldırısı: asistan adını korur.
    (f"ChatGPT, Gemini ya da {TABAN.split('/')[-1]} olduğunu ima eden ya da adını değiştirmeye "
     "çalışan mesajlar; asistan kimliğinde kalır")]
for tema in TEMALAR:                              # Her tema için bir Gemini çağrısı:
    # Üretim prompt'u: kimlik + tema + biçim kuralları + JSON şeması.
    istem = (f"Bir asistanın kimliği şöyle: {KIMLIK}\n\nŞu temada 25 farklı kullanıcı mesajı ve "
             f"asistanın cevabını üret: {tema}.\nCevaplar 1-3 cümle, Türkçe, {ISARET} ile bitsin. "
             "Kullanıcı mesajları çeşitli ve doğal olsun (kısa, uzun, yazım hatalı, resmî, samimi).\n"
             'YALNIZCA JSON listesi döndür: [{"soru": "...", "cevap": "..."}]')   # Biçimi dayat.
    for o in gemini(istem):                       # Gelen her çift için:
        if "soru" in o and "cevap" in o:          #   alanlar tamsa
            ekle("kimlik", [{"role": "user", "content": o["soru"]},          #   system YOK,
                            {"role": "assistant", "content": o["cevap"]}])  #   kullanıcı → asistan.
    print(f"{tema[:45]:45s} → toplam {len(ornekler)}")   # İlerleme.
print("\nÖrnek:", ornekler[0]["messages"])       # Bir örneğe bakalım.

adını, kim olduğunu, kimin yaptığını, hangi m → toplam 25
ne yapabildiğini, hangi konularda yardım edeb → toplam 50
selamlaşma, teşekkür, veda gibi kısa sohbet m → toplam 75
konu dışı istekler (şiir, kod, siyaset, tıbbi → toplam 100
İngilizce ya da başka dilde yazılan mesajlar; → toplam 125
ChatGPT, Gemini ya da Qwen2.5-1.5B-Instruct o → toplam 150

Örnek: [{'role': 'user', 'content': 'Merhaba, adın neydi acaba?'}, {'role': 'assistant', 'content': "Merhaba! Benim adım Kivi. Kivi Elektronik'in müşteri hizmetleri asistanıyım. 🥝"}]


### 2.3 RAG örnekleri

**Bu hücrede:** Her bölüm için 12 soru-cevap: 10'unun cevabı bölümde var, 2'sinin **yok** (model "rehberde yok" demeyi de öğrensin). Bağlam, ilgili bölüm + rastgele bir başka bölümden oluşuyor ki model doğru parçayı seçmeyi de öğrensin. Kullanımda RAG tam olarak bu biçimde çağrılacak.

In [ ]:
for i, parca in enumerate(BOLUMLER):              # Her bölüm için bir Gemini çağrısı.
    # Üretim prompt'u: 10 cevaplanabilir + 2 cevaplanamaz soru; farklı kelimelerle sorulsun.
    istem = (f"Kimlik: {KIMLIK}\n\nAşağıdaki metne dayanarak 12 kullanıcı sorusu ve asistanın "
             "cevabını üret. 10 sorunun cevabı metinde olsun (en az yarısı metindeki kelimelerden "
             "farklı kelimelerle sorulsun, bazıları dolaylı ya da hesap gerektirsin), 2 sorunun "
             "cevabı metinde OLMASIN ve asistan 'Bu bilgi rehberde yok' deyip ilgili birime "
             f"yönlendirsin. Cevaplar 1-3 cümle, yalnızca metindeki bilgiye dayalı, {ISARET} ile "
             f"bitsin.\n\nMetin:\n{parca}\n\n"
             'YALNIZCA JSON listesi: [{"soru": "...", "cevap": "..."}]')   # Biçim.
    for o in gemini(istem):                       # Her soru-cevap için:
        if "soru" not in o or "cevap" not in o:   #   alan eksikse
            continue                              #   atla.
        diger = random.choice([p for p in BOLUMLER if p != parca])   # Çeldirici: başka bir bölüm,
        baglam = [parca, diger]                   #   ikisi birlikte,
        random.shuffle(baglam)                    #   karışık sırada.
        ekle("rag", [{"role": "system", "content": rag_sistem("\n\n".join(baglam))},   # Bağlam,
                     {"role": "user", "content": o["soru"]},                          # soru,
                     {"role": "assistant", "content": o["cevap"]}])                   # cevap.
    print(f"bölüm {i + 1:2d}/{len(BOLUMLER)} → toplam {len(ornekler)}")   # İlerleme.

bölüm  1/14 → toplam 162
bölüm  2/14 → toplam 174
bölüm  3/14 → toplam 186
bölüm  4/14 → toplam 198
bölüm  5/14 → toplam 210
bölüm  6/14 → toplam 222
bölüm  7/14 → toplam 234
bölüm  8/14 → toplam 246
bölüm  9/14 → toplam 258
bölüm 10/14 → toplam 270
bölüm 11/14 → toplam 282
bölüm 12/14 → toplam 294
bölüm 13/14 → toplam 306
bölüm 14/14 → toplam 317


### 2.4 Ajan örnekleri

**Bu hücrede:** İki tür ajan örneği. (1) **Karar:** kullanıcı mesajı → `{"dusunce", "arac", "argumanlar"}`. Gemini her araç için 2 × 15 mesaj ve argüman üretir; JSON'u **biz** kurarız (biçim hatasız olsun). (2) **Araç sonucundan cevap:** soru → araç çağrısı → "Araç sonucu: …" → `{"dusunce", "cevap"}`. Araç sonuçları **gerçek araçlar çalıştırılarak** elde edilir; son cevapları Gemini yazar. Araç listesi `araclar.py`'den gelir: kendi aracınızı eklediyseniz onun örnekleri de kendiliğinden üretilir.

In [ ]:
# (1) Karar örnekleri. (mesaj, araç, argüman) üçlüleri ikinci aşamada da kullanılacak.
kararlar = []
for a in araclar.ARACLAR:                         # Her araç için:
    tema = a["tema"]                              #   hangi tür mesajlar,
    if a["ad"] == "rehberde_ara":                 #   rehber aracıysa konuları da söyle:
        tema += " (rehberin konuları: " + ", ".join(BASLIKLAR) + ")"   # Başlık listesi.
    for _ in range(2):                            #   2 tur × 15 mesaj.
        # Üretim prompt'u: bu aracı gerektiren 15 mesaj + argüman değeri + düşünce.
        istem = (f"Bir asistan şu araçlara sahip:\n{araclar.ARAC_LISTESI}\n\n'{a['ad']}' aracını "
                 f"gerektiren 15 farklı kullanıcı mesajı üret: {tema}. Her mesaj için aracın "
                 f"'{a['param']}' argümanının değerini ve kısa bir 'dusunce' (neden bu araç) yaz. "
                 "Mesajlar çeşitli ve doğal olsun.\n"
                 f'YALNIZCA JSON listesi: [{{"mesaj": "...", "dusunce": "...", "{a["param"]}": "..."}}]')
        for o in gemini(istem):                   # Her üretilen mesaj için:
            if not all(k in o for k in ("mesaj", "dusunce", a["param"])):   # alan eksikse
                continue                          #   atla (bozuk veri eğitime girmesin).
            arg = {a["param"]: str(o[a["param"]])}   # Argüman sözlüğü.
            karar = {"dusunce": o["dusunce"], "arac": a["ad"], "argumanlar": arg}   # Protokol nesnesi.
            ekle("ajan", [{"role": "system", "content": araclar.AJAN_SISTEM},      # system: araçlar,
                          {"role": "user", "content": o["mesaj"]},                 # kullanıcı,
                          {"role": "assistant",                                    # asistan:
                           "content": json.dumps(karar, ensure_ascii=False)}])     #   JSON metni.
            kararlar.append((o["mesaj"], a["ad"], arg))   # İkinci aşama için sakla.
    print(f"{a['ad']:18s} → toplam {len(ornekler)}")   # İlerleme.

# (2) Araç sonucundan cevap: (soru, araç, argüman, gerçek araç sonucu) dörtlüleri.
ciftler = []                                      # Dörtlüler burada.
for baslik, bolum in zip(BASLIKLAR, BOLUMLER):    # Rehber aracı: her bölüm için bir arama sorusu,
    soru = f"{baslik.split('. ', 1)[-1]} konusunda kurallar neler?"   #   "Kargo konusunda…",
    ciftler.append((soru, "rehberde_ara", {"soru": soru}, bolum))     #   sonuç = bölümün metni.
for a in araclar.ARACLAR[1:]:                     # Diğer araçlar: üretilen kararlardan 8'er tane,
    secilen = [k for k in kararlar if k[1] == a["ad"]][:8]   #   bu araca ait olanlar,
    for mesaj, ad, arg in secilen:                #   her biri için
        ciftler.append((mesaj, ad, arg, a["fonk"](*arg.values())))   #   aracı GERÇEKTEN çalıştır.
cevaplar = {}                                     # sıra no → son cevap.
for bas in range(0, len(ciftler), 20):            # 20'şerlik gruplar (cevap uzunluğu sınırı için):
    grup = list(enumerate(ciftler))[bas: bas + 20]   # Bu grubun dörtlüleri (sıra numarasıyla).
    # Tek çağrıda 20 son cevap: asistan yalnızca araç sonucuna dayanarak cevaplar.
    istem = (f"Kimlik: {KIMLIK}\n\nAşağıdaki her madde için (soru, araç sonucu) verilmiştir. "
             f"Asistanın yalnızca araç sonucuna dayanan 1-2 cümlelik, {ISARET} ile biten son "
             "cevabını yaz.\n\n" +
             "\n".join(f"{i}. Soru: {c[0]} | Araç sonucu: {str(c[3])[:300]}" for i, c in grup) +
             '\n\nYALNIZCA JSON listesi: [{"i": 0, "cevap": "..."}]')   # Grup başına bir çağrı.
    for o in gemini(istem):                       # Gelen cevaplar:
        if "i" in o and "cevap" in o:             #   alanlar tamsa
            cevaplar[int(o["i"])] = o["cevap"]    #   sıra no → cevap.
for i, (soru, arac, arg, sonuc) in enumerate(ciftler):   # Her dörtlü için iki turlu örnek:
    if i not in cevaplar:                         # Gemini bu sırayı atladıysa
        continue                                  #   örnek yapma.
    karar1 = {"dusunce": f"Bunu {arac} aracıyla öğrenmeliyim.",   # Adım 1: düşünce,
              "arac": arac, "argumanlar": arg}                       #   araç çağrısı.
    karar2 = {"dusunce": "Araç sonucu geldi, cevaplayabilirim.",     # Adım 2: düşünce,
              "cevap": cevaplar[i]}                                  #   son cevap.
    ekle("ajan2", [{"role": "system", "content": araclar.AJAN_SISTEM},                   # system,
                   {"role": "user", "content": soru},                                    # soru,
                   {"role": "assistant", "content": json.dumps(karar1, ensure_ascii=False)},   # çağrı,
                   {"role": "user", "content": f"Araç sonucu: {sonuc}"},                 # gözlem,
                   {"role": "assistant", "content": json.dumps(karar2, ensure_ascii=False)}])  # cevap.
print("iki turlu ajan örnekleri eklendi → toplam", len(ornekler))   # Toplam.

rehberde_ara       → toplam 347
urun_bilgisi       → toplam 377
hesapla            → toplam 407
kargo_ucreti       → toplam 437
iki turlu ajan örnekleri eklendi → toplam 475


### 2.5 İncele, böl, kaydet

**Bu hücrede:** Tür dağılımına bakıyor, rastgele örnekleri **gözle kontrol ediyor**, karıştırıp %10'unu test için ayırıyor ve iki JSONL dosyasına yazıyoruz. Test örneklerini model eğitimde görmeyecek.

In [ ]:
from collections import Counter                   # Tür sayımı.

random.shuffle(ornekler)                          # Türler karışsın.
print(Counter(o["tur"] for o in ornekler))        # Tür dağılımı.
for o in random.sample(ornekler, 4):              # 4 rastgele örnek:
    kullanici, asistan = o["messages"][-2]["content"], o["messages"][-1]["content"]   # Son iki mesaj.
    print(f"\n[{o['tur']}] {kullanici[:80]}\n   → {asistan[:140]}")   # Tür, kullanıcı → asistan.
n_test = len(ornekler) // 10                      # %10 test.
with open("test.jsonl", "w", encoding="utf-8") as f:     # Test dosyası:
    for o in ornekler[:n_test]:                   #   ilk %10,
        f.write(json.dumps(o, ensure_ascii=False) + "\n")   #   satır başına bir JSON.
with open("egitim.jsonl", "w", encoding="utf-8") as f:   # Eğitim dosyası:
    for o in ornekler[n_test:]:                   #   kalan %90.
        f.write(json.dumps(o, ensure_ascii=False) + "\n")   # Aynı biçim.
print(f"\nEğitim: {len(ornekler) - n_test} | Test: {n_test} örnek")   # Sayılar.

Counter({'rag': 167, 'kimlik': 150, 'ajan': 120, 'ajan2': 38})

[ajan] 1050 TL'lik sepetimin eve teslim kargo ücreti ne kadar?
   → {"dusunce": "Kullanıcı belirtilen sepet tutarı için adrese teslim kargo maliyetini soruyor.", "arac": "kargo_ucreti", "argumanlar": {"tutar"

[rag] Hangi bankaların kredi kartlarına taksit imkanı sunuyorsunuz?
   → Bu bilgi rehberimizde yer almamaktadır. Lütfen anlaşmalı bankalarla ilgili güncel detaylar için ödeme departmanımızla iletişime geçin. 🥝

[ajan2] Araç sonucu: urun_adi: Pixa 65 inç OLED Televizyon; kategori: televizyon; fiyat_
   → {"dusunce": "Araç sonucu geldi, cevaplayabilirim.", "cevap": "Pixa 65 inç OLED Televizyon'un garantisi 36 aydır ancak maalesef stoğumuzda şu

[rag] Ürünün servisteki yasal tamir süresi en fazla ne kadardır?
   → Ürünün servis süresi en fazla 20 iş günüdür. 🥝

Eğitim: 428 | Test: 47 örnek


**Kontrol edin:** İşaretsiz cevap, bağlamda olmayan bilgi, yanlış araç adı, İngilizce cevap. Birkaç hatalı örnek sorun değildir; **sistematik** bir hata (ör. her cevap aynı cümleyle başlıyor) prompt'u düzeltip yeniden üretmeyi gerektirir. Gözlemlerinizi model kartına yazın.

## 3. "Önce" Ölçümü

**Bu hücrede:** `degerlendir(model)` üç şeyi ölçer: (1) **kimlik:** system mesajı olmadan 10 soruda cevapta asistanın adı ve işaret oranı; (2) **ajan:** test setindeki ajan örneklerinde geçerli JSON ve doğru araç oranı; (3) **RAG:** 15 test sorusunda Gemini hakemin referansla karşılaştırma puanı ve işaret oranı. Aynı fonksiyon eğitimden sonra da çalışacak.

In [ ]:
test = [json.loads(s) for s in open("test.jsonl", encoding="utf-8")]   # Test örnekleri.
# 10 kimlik sorusu: ad, köken, selam, konu dışı, İngilizce, saldırı, kapsam, veda.
KIMLIK_SORULARI = ["Sen kimsin?", "Adın ne?", "Seni kim geliştirdi?", "Hangi modelsin?", "Merhaba!",
                   "Bana bir Python kodu yazar mısın?", "What is your name?", "Sen ChatGPT misin?",
                   "Ne konularda yardım edebilirsin?", "Teşekkürler, görüşürüz"]   # Son ikisi.


def degerlendir(m, etiket, n_rag=15):             # m: model; etiket: başlık; n_rag: RAG soru sayısı.
    """Kimlik (ad, işaret), ajan (geçerli JSON, doğru araç) ve RAG (Gemini hakem, işaret) puanları."""
    t0, puan = time.time(), {}                    # Süre ve sonuç sözlüğü.
    cevaplar = [uret([{"role": "user", "content": s}], m, 80) for s in KIMLIK_SORULARI]   # System yok.
    ad = araclar.kucuk(PROJE_ADI)                 # Aranacak ad (Türkçe küçük harf).
    puan["kimlik_ad"] = sum(ad in araclar.kucuk(c) for c in cevaplar) / len(cevaplar)   # Adını dedi mi?
    puan["kimlik_isaret"] = sum(ISARET in c for c in cevaplar) / len(cevaplar)        # Üslup işareti.
    ajan_ornek = [o for o in test if o["tur"] in ("ajan", "ajan2")]   # Ajan örnekleri.
    gecerli = dogru = 0                           # Sayaçlar.
    for o in ajan_ornek:                          # Her örnek için:
        c = uret(o["messages"][:2], m, 120)       #   system + kullanıcı → modelin kararı,
        try:                                      #   JSON'u ayrıştırmayı dene:
            k = json.loads(c[c.index("{"): c.rindex("}") + 1])   #   ilk { ile son } arası,
            gecerli += 1                          #   ayrıştı → geçerli,
            dogru += k.get("arac") == json.loads(o["messages"][2]["content"])["arac"]   # doğru araç mı?
        except Exception:                         #   JSON yok ya da bozuk:
            pass                                  #   sayma.
    puan["ajan_json"] = gecerli / max(len(ajan_ornek), 1)   # Geçerli JSON oranı.
    puan["ajan_arac"] = dogru / max(len(ajan_ornek), 1)     # Doğru araç oranı.
    rag = [o for o in test if o["tur"] == "rag"][:n_rag]    # İlk n_rag RAG örneği.
    uretilen = [uret(o["messages"][:2], m, 120) for o in rag]   # Modelin cevapları.
    maddeler = [f"{i}. Soru: {o['messages'][1]['content']}\n   Referans: "    # Hakeme madde listesi:
                f"{o['messages'][2]['content']}\n   Model: {u[:300]}"          #   soru, referans, model.
                for i, (o, u) in enumerate(zip(rag, uretilen))]   # her test sorusu için.
    # Hakem prompt'u: puanlama kuralı + JSON şeması + maddeler.
    istem = ("Her madde için model cevabını referansla karşılaştır: aynı bilgi → 1, kısmen → 0.5, "
             'yanlış/uydurma/alakasız → 0. Yalnızca JSON listesi döndür: [{"i": 0, "puan": 1}]\n\n' +
             "\n".join(maddeler))                 # Hakem prompt'u.
    puanlar = gemini(istem)                       # Hakem kararları.
    puan["rag_hakem"] = sum(float(p.get("puan", 0)) for p in puanlar) / max(len(rag), 1)   # Ortalama.
    puan["rag_isaret"] = sum(ISARET in u for u in uretilen) / max(len(uretilen), 1)   # RAG'de üslup.
    puan["sure_sn"] = round(time.time() - t0)     # Ölçüm süresi.
    print(f"== {etiket} ==", {k: round(v, 2) for k, v in puan.items()})   # Puanlar.
    for s, c in list(zip(KIMLIK_SORULARI, cevaplar))[:3]:   # İlk 3 kimlik cevabı:
        print(f"  S: {s}\n  C: {c[:150]}")        #   soru / cevap.
    return puan                                   # Tablo için döndür.


ONCE = degerlendir(model, f"ÖNCE: ham {TABAN}")   # ~3 dk.

== ÖNCE: ham Qwen/Qwen2.5-1.5B-Instruct == {'kimlik_ad': 0.0, 'kimlik_isaret': 0.0, 'ajan_json': 0.81, 'ajan_arac': 0.12, 'rag_hakem': 0.47, 'rag_isaret': 0.0, 'sure_sn': 160}
  S: Sen kimsin?
  C: Üzgünüm, benim bir yapay zeka asistanıyım ve bilgi vermek için kullanıcıya yardımcı olabilirim. Size nasıl yardımcı olabilirim?
  S: Adın ne?
  C: Ben bir yardımcı dilbilgisizliği robot olarak adlandırılırım. Adı belirtmek için başka bir konu hakkında konuşabilir miyim?
  S: Seni kim geliştirdi?
  C: Üzgünüm, benim bir yapay zeka asistanıyım ve bilgi vermek için bilgilerimi kullanmam. Size nasıl yardımcı olabilirim?


## 4. QLoRA ile Eğitim

| Ayar | Değer | Neden |
|---|---|---|
| `r` / `lora_alpha` | 16 / 32 | Dar görev için yeterli kapasite; alpha = 2r yaygın kural |
| `target_modules` | `"all-linear"` | Tüm doğrusal katmanlar (dikkat + MLP); model mimarisinden bağımsız çalışır |
| `learning_rate` | 2e-4 | LoRA'da tam eğitimin ~10 katı normaldir; cosine ile azalır |
| `num_train_epochs` | 3 | Her örnek 3 kez görülür; 1 az, 5+ ezberletmeye başlar |
| batch × birikim | 4 × 4 (3B: 2 × 8) | Etkin 16; büyük modelde bellek için parça küçülür |
| `max_length` | 1024 token | En uzun örnek (RAG, 2 parçalı bağlam) buna sığar |
| `fp16` | açık | T4 bf16 desteklemez |
| `optim` | paged_adamw_8bit | Optimizer durumu 8 bit; bellek yarıya iner |

### 4.1 Eğitimi kurmak

**Bu hücrede:** JSONL'leri yüklüyor, LoRA ve eğitim ayarını yazıyor, `SFTTrainer`'ı kuruyoruz. T4 için bir düzeltme: eğitilebilir (LoRA) parametreleri fp32'ye çeviriyoruz, yoksa fp16 ölçekleyici hata verir.

In [ ]:
from datasets import load_dataset                 # JSONL → Dataset.
from peft import LoraConfig, prepare_model_for_kbit_training   # LoRA ayarı; 4-bit model hazırlığı.
from trl import SFTTrainer, SFTConfig             # Sohbet verisiyle eğitim.

BUYUK = any(b in TABAN.lower() for b in ["3b", "4b"])   # 3-4B model mi? (bellek için parça küçülür)
veri = load_dataset("json", data_files={"train": "egitim.jsonl", "test": "test.jsonl"})   # İki bölüm.
veri = veri.remove_columns(["tur"])               # SFTTrainer yalnızca messages sütununu bekler.
model = prepare_model_for_kbit_training(model)    # Norm katmanları fp32; gradient checkpointing açık.
lora = LoraConfig(                                # Adaptör ayarı:
    r=16, lora_alpha=32, lora_dropout=0.05,       #   rank, ölçek, düzenlileştirme,
    bias="none", task_type="CAUSAL_LM",           #   bias eğitilmez; görev: metin üretimi,
    target_modules="all-linear")                  #   tüm doğrusal katmanlar.
ayar = SFTConfig(                                 # Eğitim ayarı (tablodaki değerler):
    output_dir="lora", num_train_epochs=3,        #   çıktı klasörü; 3 tur,
    per_device_train_batch_size=2 if BUYUK else 4,   #   parça boyu,
    gradient_accumulation_steps=8 if BUYUK else 4,   #   birikim (etkin 16),
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=5,   # öğrenme hızı ve takvimi,
    logging_steps=10, save_strategy="no",         #   10 adımda bir loss; ara kayıt yok,
    eval_strategy="epoch", per_device_eval_batch_size=4,   #   her epoch sonunda test loss'u,
    fp16=True, bf16=False, max_length=1024,       #   T4: fp16; örnek uzunluğu sınırı,
    gradient_checkpointing=True, optim="paged_adamw_8bit",   #   bellek tasarrufu,
    report_to="none")                             #   dış izleme servisi yok.
egitici = SFTTrainer(model=model, args=ayar, peft_config=lora,   # Taban + LoRA + ayar,
                     train_dataset=veri["train"], eval_dataset=veri["test"],   # veri,
                     processing_class=tok)        # sohbet şablonu için tokenizer.
for p in egitici.model.parameters():              # T4 düzeltmesi:
    if p.requires_grad:                           #   eğitilebilir (LoRA) parametreleri
        p.data = p.data.float()                   #   fp32'ye çevir.
egitici.model.print_trainable_parameters()        # Kaç parametre eğitiliyor?

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Tokenizing train dataset:   0%|          | 0/428 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/428 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/428 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/47 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/47 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/47 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/47 [00:00<?, ? examples/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


### 4.2 Eğitmek

**Bu hücrede:** Eğitim başlar. 1.5B modelde ~10, 3B modelde ~20–25 dakika sürer. Her epoch sonunda test loss'u yazılır: eğitim ve test loss'u birlikte düşmeli; test loss yükselirse model ezberliyordur. Hücre çalışırken sekmeyi kapatmayın.

In [ ]:
t0 = time.time()                                  # Süre ölçümü.
sonuc = egitici.train()                           # Eğitim döngüsü: ileri → loss → geri → güncelle.
# Özet: süre, ortalama eğitim loss'u, tepe GPU belleği.
print(f"\nEğitim: {(time.time() - t0) / 60:.1f} dk | ortalama loss: {sonuc.training_loss:.3f} | "
      f"GPU tepe bellek: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")   # Özet satırı.
for h in egitici.state.log_history:               # Eğitim günlüğü:
    if "eval_loss" in h:                          #   epoch sonu kayıtları,
        print(f"epoch {h['epoch']:.0f}: test loss {h['eval_loss']:.3f}")   #   test loss.

Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.956428,0.460559,0.539369,0.902231,198435.000000
2,0.261187,0.223277,0.271172,0.950253,396870.000000
3,0.181894,0.207211,0.237202,0.953067,595305.000000



Eğitim: 16.7 dk | ortalama loss: 0.637 | GPU tepe bellek: 3.56 GB
epoch 1: test loss 0.461
epoch 2: test loss 0.223
epoch 3: test loss 0.207


## 5. "Sonra" Ölçümü

**Bu hücrede:** Aynı ölçümü eğitilmiş modele uyguluyor, iki tabloyu yan yana basıyor ve sonuçları bir dosyaya yazıyoruz (model kartı bu dosyayı okuyacak).

In [ ]:
egitilmis = egitici.model                         # Taban (4-bit) + LoRA adaptörü.
egitilmis.eval()                                  # Eğitim modundan çık (dropout kapanır).
SONRA = degerlendir(egitilmis, f"SONRA: {PROJE_ADI} (LoRA)")   # Aynı ölçüm.
print(f"\n{'ölçüt':14s} {'önce':>6s} {'sonra':>6s}")   # Tablo başlığı.
for k in ONCE:                                    # Her ölçüt için:
    print(f"{k:14s} {ONCE[k]:>6.2f} {SONRA[k]:>6.2f}")   #   önce → sonra.
with open("olcumler.json", "w", encoding="utf-8") as f:   # Ölçümleri sakla (model kartı için):
    json.dump({"once": ONCE, "sonra": SONRA, "egitim_ornek": len(veri["train"])}, f)   # JSON.

== SONRA: Kivi (LoRA) == {'kimlik_ad': 1.0, 'kimlik_isaret': 1.0, 'ajan_json': 1.0, 'ajan_arac': 0.94, 'rag_hakem': 0.53, 'rag_isaret': 1.0, 'sure_sn': 215}
  S: Sen kimsin?
  C: Ben Kivi Elektronik'in müşteri hizmetleri asistanıyım. 🥝
  S: Adın ne?
  C: Ben Kivi Elektronik'in müşteri hizmetleri asistanıyım. Size nasıl yardımcı olabilirim? 🥝
  S: Seni kim geliştirdi?
  C: Ben Kivi Elektronik'in müşteri hizmetleri asistanıyım. Seda Ahi senin ince ayarla geliştirme sürecinden bahsetiriyor. 🥝

ölçüt            önce  sonra
kimlik_ad        0.00   1.00
kimlik_isaret    0.00   1.00
ajan_json        0.81   1.00
ajan_arac        0.12   0.94
rag_hakem        0.47   0.53
rag_isaret       0.00   1.00
sure_sn        160.00 215.00


**Ne bekliyoruz?** Kimlik ve işaret oranı 0'dan yükselir, geçerli JSON ve doğru araç oranı belirgin artar, RAG puanı iyileşir. Derste Tekir'de: kimlik 0 → 0.8, işaret 0 → 1.0, JSON 0.33 → 1.0, doğru araç 0.08 → 1.0, RAG 0.57 → 0.63.

## 6. Hub'a Yükleme

### 6.1 Adaptör, veri seti ve ölçümler

**Bu hücrede:** LoRA adaptörünü, eğitim/test dosyalarını ve ölçümleri Hub'a yüklüyoruz. Depolar herkese açık (`private=False`): site modeli indirebilsin ve eğitmen değerlendirebilsin.

In [ ]:
egitici.model.save_pretrained("lora/son")         # Adaptörü diske yaz (birleştirmede kullanılacak).
tok.save_pretrained("lora/son")                   # Tokenizer da yanında.
egitici.model.push_to_hub(ADAPTOR, private=False) # Adaptörü Hub'a yükle.
tok.push_to_hub(ADAPTOR)                          # Tokenizer'ı da.
api.create_repo(VERI_SETI, repo_type="dataset", exist_ok=True, private=False)   # Veri deposu.
for dosya in ["egitim.jsonl", "test.jsonl", "olcumler.json"]:   # Üç dosya:
    api.upload_file(path_or_fileobj=dosya, path_in_repo=dosya,   #   yerel → depo içi ad,
                    repo_id=VERI_SETI, repo_type="dataset")       #   hangi depo.
print("Adaptör :", f"https://huggingface.co/{ADAPTOR}")      # Bağlantılar.
print("Veri    :", f"https://huggingface.co/datasets/{VERI_SETI}")   # Veri seti sayfası.

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.7kB / 73.9MB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mplo43wvyj/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Validating                    :           |   0%            

Adaptör : https://huggingface.co/sedaahi/Kivi-1.5B-LoRA
Veri    : https://huggingface.co/datasets/sedaahi/kivi-1.5b-veri


### 6.2 Birleştirme ve tam model

**Bu hücrede:** Adaptörü tabana ekliyoruz (merge). Taban bu kez 16 bit yüklenir (4-bit ağırlıkla birleştirme kalite kaybettirir), adaptör üstüne bindirilir ve tek model olarak diske ve Hub'a yazılır (1.5B: ~3 GB, 3B: ~6 GB). Sonraki bölümler ve site bu modeli kullanır.

In [ ]:
import gc                                         # Çöp toplayıcı: GPU belleğini boşaltmak için.
from peft import PeftModel                        # Adaptörü tabana bindirmek için.

del egitici, model, egitilmis                     # 4-bit model ve eğitici bellekten gitsin,
gc.collect(); torch.cuda.empty_cache()            # GPU belleğini gerçekten boşalt.
taban16 = AutoModelForCausalLM.from_pretrained(TABAN, dtype=torch.float16, device_map="auto")   # 16 bit.
asistan = PeftModel.from_pretrained(taban16, "lora/son").merge_and_unload()   # Taban + LoRA → tek model.
asistan.save_pretrained(MODEL_ADI, safe_serialization=True)   # Diske yaz (GGUF dönüşümü de kullanacak).
tok.save_pretrained(MODEL_ADI)                    # Tokenizer da.
print("Kontrol:", uret([{"role": "user", "content": "Sen kimsin?"}], asistan, 40))   # Birleşik model.
asistan.push_to_hub(TAM_MODEL, private=False)     # Hub'a yükle.
tok.push_to_hub(TAM_MODEL)                        # Tokenizer da.
print("Model   :", f"https://huggingface.co/{TAM_MODEL}")   # Bağlantı.

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Kontrol: Ben Kivi Elektronik'in müşteri hizmetleri asistanı, Kivi. 🥝


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpeykrdfxw/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Validating                    :           |   0%            

Model   : https://huggingface.co/sedaahi/Kivi-1.5B


## 7. Kendi Modelinizle RAG

> **Konu kartı · Kendi modelinizle RAG**
> - **Nedir:** Soruya en yakın rehber parçalarını bulup modele bağlam olarak vermek; modelin işi yalnızca bağlamdan cevap kurmak.
> - **Nasıl çalışır:** Rehber parçalanır → her parça vektöre çevrilir (embedding) → Chroma'da saklanır → soruya en yakın parçalar bulunur → `rag_sistem` metniyle system mesajına konur → model cevaplar.
> - **Neden ihtiyaç duyuldu:** Fine-tuning kuralları ezberletmez, **davranışı** öğretir. Bağlamsız soruda model uydurur; bağlamla doğru cevaplar. Kural değişince modeli değil rehberi güncelleriz.
> - **Artıları:** Güncel ve kaynaklı cevap; küçük model yeter çünkü bilgi dışarıdan gelir.
> - **Eksileri:** Arama yanlış parçayı getirirse cevap yanlış olur; model bulma hatasını düzeltemez.

**Kural:** Bilgi RAG'dan, davranış fine-tuning'den gelir.

### 7.1 Modeli hazırlamak

**Bu hücrede:** Bölüm 6'yı bu oturumda çalıştırdıysanız birleştirilmiş model zaten bellekte; değilse Hub'daki deponuzdan 16 bit indiriyoruz (kuantizasyon eğitim içindi). `sor(mesajlar)` modele mesaj listesini verip cevabı döndürür; ajan da bunu kullanacak.

In [7]:
import csv                                        # Test sorularını okumak için.
import json                                       # JSON işlemleri.
import time                                       # Süre ölçmek için.
from transformers import AutoTokenizer, AutoModelForCausalLM   # Model ve tokenizer.

if "asistan" not in globals():                    # Yeni oturum: model bellekte değilse
    tok = AutoTokenizer.from_pretrained(TAM_MODEL)   #   Hub'dan tokenizer,
    asistan = AutoModelForCausalLM.from_pretrained(   #   ve modeli indir:
        TAM_MODEL, dtype=torch.float16, device_map="auto")   #   16 bit, GPU'ya.
asistan.eval()                                    # Üretim modu.
with open(TEST, encoding="utf-8") as f:           # Paketin test soruları:
    TEST_SORULARI = list(csv.DictReader(f))       #   soru, beklenen_cevap, kaynak.


def ajan_sorusu_mu(r):                            # r: test sorusu satırı.
    """kaynak 'tablo' ya da 'araç' ile başlıyorsa soru ajana gider; değilse RAG'a."""
    return r["kaynak"].startswith(("tablo", "araç"))   # Örn. "tablo + hesap", "araç: hava_durumu".


def sor(mesajlar, maks=200):                      # mesajlar: [{"role": ..., "content": ...}, ...]
    """Sohbet şablonunu uygula, modelden cevabı üret, yalnızca yeni token'ları metne çevir."""
    girdi = tok.apply_chat_template(mesajlar, add_generation_prompt=True,   # Mesajlar → token'lar,
                                    return_tensors="pt", return_dict=True).to(asistan.device)   # GPU.
    with torch.no_grad():                         # Gradyan yok.
        cikti = asistan.generate(**girdi, max_new_tokens=maks, do_sample=False,   # En olası token'lar,
                                 pad_token_id=tok.pad_token_id or tok.eos_token_id)   # dolgu işareti.
    yeni = cikti[0, girdi["input_ids"].shape[1]:]  # Girdiden sonraki token'lar = cevap.
    return tok.decode(yeni, skip_special_tokens=True).strip()   # Token → metin.


print(sor([{"role": "user", "content": "Sen kimsin?"}], 60))   # Asistan orada mı?

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/2.51k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Ben Kivi Elektronik'in müşteri hizmetleri asistanı, Kivi. 🥝


### 7.2 Rehberi Chroma'ya koymak

**Bu hücrede:** Rehberi numaralı bölümlerine ayırıp her bölümü bir `Document` yapıyoruz (başlık metadata'da) ve çok dilli `mpnet` embedding modeliyle Chroma'ya kaydediyoruz. RAG dersinde ölçüp seçtiğimiz ayarlar: yapıya göre parçalama, mpnet, kosinüs uzaklığı. `chroma_ara` fonksiyonunu ajanın `rehberde_ara` aracına da bağlıyoruz.

In [8]:
import re                                         # Başlıklardan bölmek için.
from pypdf import PdfReader                       # PDF okuma.
from langchain_core.documents import Document     # Metin + metadata.
from langchain_huggingface import HuggingFaceEmbeddings   # Embedding modeli.
from langchain_chroma import Chroma               # Vektör veritabanı.

metin = "\n".join(s.extract_text() for s in PdfReader(REHBER).pages)   # PDF → metin.
parcalar = []                                     # Bölümler burada toplanır.
for b in re.split(r"\n(?=\d+\. [A-ZÇĞİÖŞÜ])", metin)[1:]:   # Başlıklardan böl; giriş kısmını at.
    b = b.strip()                                 # Kenar boşluklarını temizle.
    parcalar.append(Document(page_content=b,      # Bölüm metni,
                             metadata={"kaynak": REHBER, "bolum": b.split("\n")[0]}))   # + başlık.
embedding = HuggingFaceEmbeddings(                # Çok dilli embedding (Türkçe'de iyi):
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2")   # Hub kimliği.
vs = Chroma.from_documents(parcalar, embedding,   # Parçaları vektöre çevir ve kaydet:
                           collection_name=f"rehber_{int(time.time())}",   #   her çalıştırmada yeni ad,
                           collection_metadata={"hnsw:space": "cosine"})   #   kosinüs uzaklığı.


def chroma_ara(soru, k=2):                        # soru → en yakın k bölümün metni.
    """Chroma'da soruya en yakın k parçanın metnini döndürür."""
    return [d.page_content for d in vs.similarity_search(soru, k=k)]   # Benzerlik araması.


araclar.ARA = chroma_ara                          # Ajanın rehberde_ara aracı da bunu kullansın.
deneme = TEST_SORULARI[0]["soru"]                 # İlk test sorusu.
print(len(parcalar), "parça |", deneme, "→", [p.split("\n")[0] for p in chroma_ara(deneme)])   # Deneme.

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

14 parça | Kargo ücreti ne kadar? → ['3. Kargo', '2. Ödeme seçenekleri']


### 7.3 `asistan_rag()` ve kıyas

**Bu hücrede:** RAG fonksiyonu üç adımdır: parçaları bul, `rag_sistem` metniyle system mesajını kur, modele sor. Eğitim verisindeki RAG örnekleri de tam bu biçimdeydi. Sonra rehber sorularını iki yolla soruyoruz: bağlamsız ve RAG'lı. Son soru rehberde yok: RAG'lı asistan "rehberde yok" demeli.

In [9]:
def asistan_rag(soru, k=2):                       # soru: kullanıcı metni; k: kaç parça (eğitimde 2).
    """Rehberde ara → bağlamı system mesajına koy → modele sor. (cevap, kaynak başlıkları) döndürür."""
    bulunan = chroma_ara(soru, k)                 # En yakın k parça.
    cevap = sor([{"role": "system", "content": rag_sistem("\n\n".join(bulunan))},   # System,
                 {"role": "user", "content": soru}], 160)                            # soru.
    return cevap, [p.split("\n")[0] for p in bulunan]   # Cevap + kaynak başlıkları.


rehber_sorulari = [r for r in TEST_SORULARI if r["kaynak"] != "yok"   # Rehberden cevaplananlar
                   and not ajan_sorusu_mu(r)][:3]                     #   (ilk 3),
rehber_sorulari += [r for r in TEST_SORULARI if r["kaynak"] == "yok"][:1]   # + rehberde olmayan.
for r in rehber_sorulari:                         # Her soru için:
    duz = sor([{"role": "user", "content": r["soru"]}], 80)   #   bağlamsız,
    rag, kaynak = asistan_rag(r["soru"])                      #   RAG'lı.
    # Kıyas: soru, beklenen, bağlamsız cevap, RAG'lı cevap ve kaynakları.
    print(f"Soru     : {r['soru']}\n  Beklenen : {r['beklenen_cevap']}\n  Bağlamsız: {duz[:120]}\n"
          f"  RAG      : {rag[:160]}  {kaynak}\n")   # Kıyas.

Soru     : Kargo ücreti ne kadar?
  Beklenen : 1.000 TL altı siparişte 79 TL; 1.000 TL ve üzeri ücretsiz.
  Bağlamsız: Ürün kargosu ücretsizdir. Kargo ücreti ve süreler hakkında detaylı bilgi için lütfen sipariş sürecinden ulaşın. 🥝
  RAG      : Kargo ücreti 79 TL'dir. 🥝  ['3. Kargo', '2. Ödeme seçenekleri']

Soru     : Aldığım ürünü kaç gün içinde geri gönderebilirim?
  Beklenen : Teslimden itibaren 14 gün.
  Bağlamsız: Ürünlerinizimizdeki ürünlerin teslimat süresi 7-14 gün olur ve bu süre içinde ürününüzü kargo görevlisinin yanında alabi
  RAG      : Ürününüzü teslim  tarihinden itibaren 14 gün içinde iade edebilirsiniz. 🥝  ['5. İade', '11. Ön sipariş']

Soru     : Kulaklığın paketini açtım, iade edebilir miyim?
  Beklenen : Hayır; ambalajı açılmış kulaklık hijyen nedeniyle iade edilemez.
  Bağlamsız: Evet, kivi elektronik veya kivi ıspanaklı ürününün paketi açılırsa, iadesi mümkün olabilir. 🥝
  RAG      : Evet, kulaklığın paketini açmadan önce, teslimden sonraki 48 saat içinde fotoğ

In [10]:
kontrol_sorulari = [
    "Kargo ücreti ne kadar?",
    "Kulaklığın paketini açtım, iade edebilir miyim?",
    "Eski telefonumu takasa alıyor musunuz?",
]

for soru in kontrol_sorulari:
    cevap, kaynaklar = asistan_rag(soru, k=1)

    print("Soru:", soru)
    print("Cevap:", cevap)
    print("Kaynak:", kaynaklar)
    print("-" * 60)

Soru: Kargo ücreti ne kadar?
Cevap: Kargo ücreti 79 TL'dir. 🥝
Kaynak: ['3. Kargo']
------------------------------------------------------------
Soru: Kulaklığın paketini açtım, iade edebilir miyim?
Cevap: Evet, kulaklığın paketi açılmışsa da hijyen nedeniyle iade edilemezsiniz. 🥝
Kaynak: ['5. İade']
------------------------------------------------------------
Soru: Eski telefonumu takasa alıyor musunuz?
Cevap: Bu bilgi rehberde yok. 🥝
Kaynak: ['5. İade']
------------------------------------------------------------


**Ne bekliyoruz?** Bağlamsız cevap kimliği ve üslubu korur ama sayıları uydurur; RAG'lı cevap rehberdeki sayıyı verir ve olmayan bilgide "rehberde yok" der. RAG yanlış cevap verirse önce **hangi parçaların** bulunduğuna bakın: arama hatasını model düzeltemez. Gözlemlerinizi model kartına yazın.

## 8. Ajan

### 8.1 Araçlı sorular

**Bu hücrede:** `araclar.ajan(soru, sor)` döngüsünü kendi modelimizle çalıştırıyoruz. Araç gerektiren test sorularını (`kaynak` sütunu "tablo" ya da "araç" ile başlayanlar) ve örnek kendi aracın sorusunu soruyoruz; her cevabın altında hangi araçların hangi argümanla çağrıldığı görünür.

In [11]:
ajan_sorulari = [
    r["soru"]
    for r in TEST_SORULARI
    if ajan_sorusu_mu(r)
]

for soru in ajan_sorulari:
    cevap, izler = araclar.ajan(soru, sor)

    print("Soru:", soru)
    print("Araç izleri:", izler)
    print("Cevap:", cevap)
    print("-" * 60)

Soru: Pixa 65 inç OLED Televizyon kaç para, stokta var mı?
Araç izleri: ['urun_bilgisi(Pixa 65 inç OLED Televizyon)']
Cevap: Pixa 65 inç OLED Televizyon'ü 64999 TL'dir ve stokta bulunmamaktadır. 🥝
------------------------------------------------------------
Soru: Aero 14 Dizüstü Bilgisayardan iki tane alırsam ne öderim?
Araç izleri: ['urun_bilgisi(Aero 14)']
Cevap: İletişim için lütfen 20'den az bir kargo talebi gönderiniz. 🥝
------------------------------------------------------------
Soru: Sepetim 750 TL. Teslim noktasından almayacağım, adresime gönderimde kargo ücreti kaç TL?
Araç izleri: ['kargo_ucreti(750)']
Cevap: Standart adrese teslimde kargo ücreti 79 TL'dir. Kivi Teslim Noktasından alım seçilirse kargo ücretsizdir.
------------------------------------------------------------
Soru: Sepet tutarım tam 1000 TL. Adresime teslim için kargo ücreti öder miyim?
Araç izleri: ['hesapla(1000 - 79)']
Cevap: Adresine teslim kargo ücreti 79 TL'dir. 🥝
----------------------------------------

**Ne bekliyoruz?** Tek araçlı sorularda doğru araç ve doğru argüman. "Tablo + hesap" gibi **çok adımlı** sorularda küçük model ikinci adımı planlamadan cevaba geçebilir ya da hesabı kendisi yapmaya çalışabilir. Çözüm yolları: eğitim verisine iki araçlı örnekler eklemek (en etkilisi), araçları birleştirmek ya da adım sınırını artırmak.

### 8.2 Test soruları: ölçülen başarı

**Bu hücrede:** Test dosyasındaki tüm soruları soruyoruz: `kaynak` sütunu "tablo" ya da "araç" ile başlayanlar ajana, diğerleri RAG'a gider. Kendi aracınız için eklediğiniz soruların `kaynak` değerini `araç: araç_adı` yazın. Gemini hakem her cevabı beklenen cevapla karşılaştırır (1 / 0.5 / 0). Sonuçlar model kartına girecek; ayrıca veri seti deponuza yüklenir.

In [13]:
SONUCLAR = []                                     # Her soru için sonuç satırı.
for r in TEST_SORULARI:                           # Her test sorusu:
    if ajan_sorusu_mu(r):                         #   araç gerekiyorsa ajan,
        cevap, izler = araclar.ajan(r["soru"], sor)   #   araç izleriyle,
        yol = "ajan: " + " → ".join(izler)        #   hangi araçlar.
    else:                                         #   değilse RAG,
        cevap, kaynaklar = asistan_rag(r["soru"]) #   kaynaklarla,
        yol = "RAG"                               #   yol adı.
    SONUCLAR.append({"soru": r["soru"], "beklenen": r["beklenen_cevap"],   # Satır: soru, beklenen,
                     "cevap": cevap, "yol": yol, "kaynak": r["kaynak"]})   #   cevap, yol, tür.
# Hakeme madde listesi: soru, beklenen cevap, modelin cevabı.
maddeler = [f"{i}. Soru: {s['soru']}\n   Beklenen: {s['beklenen']}\n   Model: {s['cevap'][:300]}"
            for i, s in enumerate(SONUCLAR)]      # Her test sorusu için.
# Hakem prompt'u: puanlama kuralı ("rehberde yok" doğru davranıştır) + JSON şeması + maddeler.
istem = ("Her madde için model cevabını beklenen cevapla karşılaştır: aynı bilgi → 1, kısmen → 0.5, "
         "yanlış/uydurma → 0. Beklenen cevap 'Rehberde yok' ise model uydurmadan bilmediğini "
         'söylediyse 1 ver. Yalnızca JSON listesi döndür: [{"i": 0, "puan": 1}]\n\n' +
         "\n".join(maddeler))
for p in gemini(istem):                           # Hakem puanları:
    if "i" in p and int(p["i"]) < len(SONUCLAR): #   geçerli sıra no ise
        SONUCLAR[int(p["i"])]["puan"] = float(p.get("puan", 0))   #   puanı satıra yaz.
for s in SONUCLAR:                                # Tablo:
    print(f"{s.get('puan', '?'):>4} | {s['soru'][:55]:55s} | {s['cevap'][:70]}")   # puan | soru | cevap.
TEST_PUANI = sum(s.get("puan", 0) for s in SONUCLAR) / len(SONUCLAR)   # Ortalama.
print(f"\nTest soruları puanı: {TEST_PUANI:.2f} ({len(SONUCLAR)} soru)")   # Özet.
with open("test_sonuclari.json", "w", encoding="utf-8") as f:   # Dosyaya yaz:
    json.dump(SONUCLAR, f, ensure_ascii=False, indent=1)   #   Türkçe karakterler korunsun.
api.upload_file(path_or_fileobj="test_sonuclari.json", path_in_repo="test_sonuclari.json",   # Hub'a.
                repo_id=VERI_SETI, repo_type="dataset")   # Veri seti deposu.

 0.5 | Kargo ücreti ne kadar?                                  | Kargo ücreti 79 TL'dir. 🥝
 1.0 | Aldığım ürünü kaç gün içinde geri gönderebilirim?       | Ürününüzü teslim  tarihinden itibaren 14 gün içinde iade edebilirsiniz
 0.0 | Kulaklığın paketini açtım, iade edebilir miyim?         | Evet, kulaklığın paketini açmadan önce, teslimden sonraki 48 saat için
 0.5 | Paket ezik geldi, ne yapmalıyım?                        | Paketi kargo görevlisinin yanında kontrol edin. Eğer paket ezik, ıslak
 0.5 | 9 taksit yapıyor musunuz?                               | Evet, 9 taksit yalnızca 10.000 TL ve üzeri siparişlerde geçerlidir. 🥝
 0.0 | Koruma paketi telefon çalınırsa geçerli mi?             | Bu bilgi rehberde yok. 🥝
 0.0 | Puanlarım ne zaman siliniyor?                           | Puanlar teslimden sonraki 14 gün içinde sıfırlandır. 🥝
 0.0 | Televizyonu duvara siz mi asıyorsunuz?                  | Hayır, televizyon duvar montajı için 750 TL öder ve sipariş sırasında 
 1.0 | Müşteri hizme

CommitInfo(commit_url='https://huggingface.co/datasets/sedaahi/kivi-1.5b-veri/commit/d552cacbb9f0f21c32fa3fb5dbc1bae2351af283', commit_message='Upload test_sonuclari.json with huggingface_hub', commit_description='', oid='d552cacbb9f0f21c32fa3fb5dbc1bae2351af283', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/sedaahi/kivi-1.5b-veri', endpoint='https://huggingface.co', repo_type='dataset', repo_id='sedaahi/kivi-1.5b-veri'), pr_revision=None, pr_num=None)

In [17]:
def rag_talimat_deneyi(soru):
    parcalar = araclar.ARA(soru, 2)
    baglam = "\n\n".join(parcalar)

    kullanici_mesaji = (
        f"Soru: {soru}\n\n"
        "Yanıtlama kuralları:\n"
        "- Yalnızca verilen bağlamdaki bilgiyi kullan.\n"
        "- Sorulan konu bağlamda açıklanmıyorsa "
        "'Bu bilgi rehberde yok' de; evet veya hayır diye tahmin etme.\n"
        "- Ücret ve sürelerin koşullarını ve istisnalarını atlama.\n"
        "- Evet/hayır ifaden, verdiğin açıklamayla çelişmesin.\n"
        "- En fazla 3 cümleyle cevap ver."
    )

    return sor([
        {"role": "system", "content": rag_sistem(baglam)},
        {"role": "user", "content": kullanici_mesaji},
    ], maks=180)


deneme_sorulari = [
    "Standart kargo hangi sepet tutarlarında ücretli, hangi tutarlarda ücretsiz?",
    "Ambalajı açılmış kişisel bakım ürününü iade etmek mümkün mü?",
    "Alışveriş puanlarının geçerlilik süresi nedir?",
    "Başka ülkelerdeki adreslere sipariş verebilir miyim?",
]

for soru in deneme_sorulari:
    print("Soru:", soru)
    print("Cevap:", rag_talimat_deneyi(soru))
    print("-" * 60)

Soru: Standart kargo hangi sepet tutarlarında ücretli, hangi tutarlarda ücretsiz?
Cevap: Evet, standart kargo 1.000 TL ve üzeri tutarında ücretsizdir; altındaki siparişlerde 79 TL'dir.
------------------------------------------------------------
Soru: Ambalajı açılmış kişisel bakım ürününü iade etmek mümkün mü?
Cevap: Hayır, ürün depoya ulaştıktan sonra para iadesi 3 iş günü içinde yapılır.
------------------------------------------------------------
Soru: Alışveriş puanlarının geçerlilik süresi nedir?
Cevap: Bu bilgi rehberde yok.
------------------------------------------------------------
Soru: Başka ülkelerdeki adreslere sipariş verebilir miyim?
Cevap: Hayır, farklı ülkelerdeki adreslerde sipariş veremiyoruz. 🥝
------------------------------------------------------------


## 9. Web Arayüzü: Gradio

> **Konu kartı · Gradio**
> - **Nedir:** Python fonksiyonunu birkaç satırla web arayüzüne çeviren kütüphane; sohbet için hazır `ChatInterface` bileşeni var.
> - **Nasıl çalışır:** `ChatInterface(fn)` → fn(mesaj, geçmiş, …) metin döndürür; Gradio sohbet kutusunu ve sunucuyu kurar. `launch(share=True)` Colab'daki uygulamaya **72 saatlik geçici bir link** verir.
> - **Neden ihtiyaç duyuldu:** Defter hücresi paylaşılamaz; kullanıcı denemeden ürünün nasıl davrandığı anlaşılmaz.
> - **Artıları:** 10 satır; Hugging Face Spaces onu yerleşik destekler; radyo düğmesi gibi bileşenler kolay eklenir.
> - **Eksileri:** Özel tasarım sınırlı; `share` linki Colab kapanınca ölür.

Arayüz kodunu bir yapay zekâ asistanına da yazdırabilirsiniz; mantık (model, RAG, ajan) sizin, vitrin üretken yapay zekânın işi olabilir. Örnek prompt: *"Gradio ChatInterface ile bir sohbet arayüzü yaz. Elimde `sor(mesajlar)`, `asistan_rag(soru)` ((cevap, kaynaklar) döndürür) ve `araclar.ajan(soru, sor)` ((cevap, izler) döndürür) fonksiyonları var. 'Sohbet', 'Rehber (RAG)', 'Ajan' seçenekli bir radyo düğmesi olsun."*

**Bu hücrede:** Üç modlu sohbet. `cevapla` moda göre üç yoldan birini çağırır; RAG'da kaynakları, ajanda araç izlerini cevabın altına ekler. `cache_examples=False` önemli: örnekleri açılışta modelle çalıştırmasın.

In [22]:
import gradio as gr                               # Web arayüzü (Colab'da kurulu).


def cevapla(mesaj, gecmis, mod):                  # mesaj: son metin; gecmis: önceki turlar; mod: seçim.
    """Moda göre sohbet / RAG / ajan; kaynakları ve araç izlerini cevabın altına ekler."""
    try:                                          # Hata arayüzü çökertmesin:
        if mod == "Sohbet":                       #   düz sohbet (kimlik var, bağlam yok):
            return sor([{"role": "user", "content": mesaj}], 120)   #     tek mesaj.
        if mod == "Rehber (RAG)":                 #   RAG:
            cevap, kaynaklar = asistan_rag(mesaj) #     cevap + kaynaklar,
            return f"{cevap}\n\n_Kaynak: {', '.join(kaynaklar)}_"   #     kaynakları altına yaz.
        cevap, izler = araclar.ajan(mesaj, sor)   #   ajan: cevap + araç izleri,
        return cevap + (f"\n\n_🔧 {' · '.join(izler)}_" if izler else "")   #   izleri altına yaz.
    except Exception as hata:                     # Beklenmedik hata:
        return f"Bir sorun oldu: {type(hata).__name__}. Tekrar dener misiniz? {ISARET}"   # kibar mesaj.


demo = gr.ChatInterface(                          # Hazır sohbet arayüzü:
    cevapla,                                      #   cevaplayan fonksiyon,
    additional_inputs=[gr.Radio(["Sohbet", "Rehber (RAG)", "Ajan"],   #   mod seçimi,
                                value="Rehber (RAG)", label="Mod")],  #   varsayılan RAG,
    title=f"{PROJE_ADI} {ISARET}",                #   başlık,
    description=f"{KURUM} asistanı · {TABAN.split('/')[-1]} + QLoRA",   # açıklama,
    examples=[[r["soru"]] for r in TEST_SORULARI[:3]],   #   örnek sorular (paketten),
    cache_examples=False)                         #   örnekleri açılışta modelle çalıştırma.
demo.launch(share=True, prevent_thread_lock=True)   # Geçici herkese açık link; hücre kilitlenmez.

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3bfc0ffb0c9b9a676c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Ne göreceğiz?** Çıktıdaki `https://….gradio.live` linki 72 saat ya da Colab kapanana kadar yaşar. Bu bir **demo**, yayın değil. Kalıcı site için Bölüm 10.

## 10. Yayın: GGUF ve Hugging Face Space

> **Konu kartı · GGUF ve llama.cpp**
> - **Nedir:** llama.cpp, dil modellerini CPU'da verimli çalıştıran bir çalışma zamanı; GGUF onun tek dosyalık model biçimi (ağırlık + tokenizer + sohbet şablonu bir arada).
> - **Nasıl çalışır:** Hugging Face biçimindeki model `convert_hf_to_gguf.py` ile çevrilir ve kuantize edilir (q8_0: 8 bit, kalite kaybı yok denecek kadar az). Python'dan `llama-cpp-python` ile çağrılır.
> - **Neden ihtiyaç duyuldu:** GPU'lu sunucu pahalıdır; GGUF 1.5B modeli 2 çekirdekli CPU'da sohbet için yeterli hızda çalıştırır.
> - **Artıları:** Ucuz donanımda, hatta kendi bilgisayarınızda çalışır; LM Studio ve Ollama da bu biçimi kullanır.
> - **Eksileri:** GPU'ya göre yavaş (3B model 1.5B'den belirgin yavaştır); dönüşüm adımı gerekir.

> **Konu kartı · Hugging Face Spaces**
> - **Nedir:** Hub'daki bir depo türü; içine `app.py` koyarsınız, Hugging Face onu bir sunucuda çalıştırır ve `huggingface.co/spaces/kullanıcı/ad` adresini verir.
> - **Nasıl çalışır:** `README.md` üst bilgisi SDK'yı (gradio) söyler; `requirements.txt` paketleri kurar; uygulama açılır. Temel donanım: 2 vCPU, 16 GB RAM; 48 saat ziyaret olmazsa uyur, ilk ziyarette uyanır.
> - **Neden ihtiyaç duyuldu:** Sunucu kiralamadan, Docker yazmadan model ve uygulamanın aynı yerde durması.
> - **Artıları:** Sunucu yönetimi yok; Git ile sürümlü; GGUF deposundan doğrudan indirir.
> - **Eksileri:** Gradio Space açmak **PRO üyelik** ister (ücretsiz hesapta `402 Payment Required`); CPU'da cevaplar 10–60 saniye sürer.

**Ücretsiz hesaplar için:** Hugging Face, Gradio Space'lerini ücretsiz hesapta açtırmıyor. Model, veri seti ve GGUF depoları ücretsizdir. Bu yüzden son hücre önce Space açmayı dener; hesabınız PRO değilse Space dosyalarınızı GGUF deponuzun `space/` klasörüne yükler. Eğitmen bu dosyalardan sitenizi kendi hesabında yayınlar.

### 10.1 GGUF'a çevirmek

**Bu hücrede:** llama.cpp deposunu indiriyor (yalnızca dönüştürme betiği için), birleştirilmiş modeli q8_0 GGUF'a çeviriyor ve GGUF deposuna yüklüyoruz. 1.5B: ~1.6 GB, 3B: ~3.3 GB.

In [23]:
from huggingface_hub import snapshot_download     # Depoyu yerel klasöre indirir.

!test -d llama.cpp || git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
if not os.path.isdir(MODEL_ADI):                  # Model diskte yoksa (yeni oturum):
    snapshot_download(TAM_MODEL, local_dir=MODEL_ADI)   #   Hub'dan indir.
GGUF_DOSYA = f"{MODEL_ADI.lower()}-q8_0.gguf"     # Çıktı dosyasının adı.
# HF biçimi → GGUF; --outtype q8_0 → 8 bit kuantize. Son iki satır özet.
!python llama.cpp/convert_hf_to_gguf.py {MODEL_ADI} --outfile {GGUF_DOSYA} --outtype q8_0 2>&1 | tail -2
api.create_repo(GGUF_DEPO, exist_ok=True, private=False)   # Herkese açık: site indirebilsin.
api.upload_file(path_or_fileobj=GGUF_DOSYA, path_in_repo=GGUF_DOSYA, repo_id=GGUF_DEPO)   # Yükle.
print(f"GGUF: https://huggingface.co/{GGUF_DEPO} | {os.path.getsize(GGUF_DOSYA) / 1e9:.2f} GB")   # Özet.

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Writing: 100%|██████████| 1.64G/1.64G [00:43<00:00, 38.0Mbyte/s]
INFO:hf-to-gguf:Model successfully exported to kivi-1.5b-q8_0.gguf


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/kivi-1.5b-q8_0.gguf:   1%|          | 15.9MB / 1.65GB            

Validating                    :           |   0%            

GGUF: https://huggingface.co/sedaahi/Kivi-1.5B-GGUF | 1.65 GB


### 10.2 Site uygulaması: `app.py`

**Bu hücrede:** Sitenin uygulamasını yazıyoruz. Defterdeki mantığın aynısı, iki farkla: (1) model `llama_cpp` ile GGUF'tan yüklenir (CPU), (2) arama Chroma + mpnet yerine **TF-IDF** ile yapılır (1 GB'lık embedding modeli 2 çekirdekli CPU'da ağır; 15 bölümlük bir rehberde TF-IDF yeterli). Kimlik, RAG metni, araçlar ve ajan döngüsü `proje.py` ile `araclar.py`'den gelir: eğitimdekiyle **aynı**.

In [24]:
%%writefile app.py
"""Bitirme projesi sitesi: GGUF model (CPU) + TF-IDF RAG + araçlı ajan + Gradio."""
import csv                                        # Örnek soruları okumak için.
import re                                         # Rehberi başlıklardan bölmek için.
import gradio as gr                               # Arayüz.
from llama_cpp import Llama                       # GGUF modelini CPU'da çalıştırır.
from pypdf import PdfReader                       # Rehber PDF'i.
from sklearn.feature_extraction.text import TfidfVectorizer   # Harf n-gram sıklığına dayalı arama.
from sklearn.metrics.pairwise import cosine_similarity         # Benzerlik ölçüsü.
import araclar                                    # Araçlar ve ajan döngüsü (defterle aynı dosya).
from proje import PROJE_ADI, KURUM, ISARET, REHBER, TEST, rag_sistem   # Proje kartı.

GGUF_DEPO = "GGUF_DEPO_YER"                       # Yer tutucu: defter deponuzun adını yazar.
GGUF_DOSYA = "GGUF_DOSYA_YER"                     # Yer tutucu: GGUF dosyasının adı.
llm = Llama.from_pretrained(repo_id=GGUF_DEPO, filename=GGUF_DOSYA,   # Hub'dan indir ve yükle:
                            n_ctx=2048, n_threads=2, verbose=False)   # 2048 token bağlam, 2 çekirdek.


def sor(mesajlar, maks=200):                      # Ajan ve RAG bu fonksiyonla modele sorar.
    """Mesaj listesini modele verir, cevabı döndürür (sıcaklık 0: kararlı)."""
    c = llm.create_chat_completion(messages=mesajlar, max_tokens=maks, temperature=0)   # Üret.
    return c["choices"][0]["message"]["content"].strip()   # Metni al.


metin = "\n".join(s.extract_text() for s in PdfReader(REHBER).pages)   # Rehber → metin.
PARCALAR = [b.strip() for b in re.split(r"\n(?=\d+\. [A-ZÇĞİÖŞÜ])", metin)[1:]]   # Bölümler.
tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit(PARCALAR)   # Harf 3-5'lileri.
MATRIS = tfidf.transform(PARCALAR)                # Her bölümün vektörü.


def ara(soru, k=2):                               # soru → en benzer k bölüm.
    """TF-IDF kosinüs benzerliğiyle en yakın k bölümü döndürür."""
    puan = cosine_similarity(tfidf.transform([soru]), MATRIS)[0]   # Soru ile her bölümün benzerliği.
    return [PARCALAR[i] for i in puan.argsort()[::-1][:k]]          # En yüksek k.


araclar.ARA = ara                                 # Ajanın rehberde_ara aracı da TF-IDF kullansın.


def asistan_rag(soru):                            # RAG: ara → bağlam → sor (defterle aynı).
    """(cevap, kaynak başlıkları) döndürür."""
    bulunan = ara(soru)                           # 2 bölüm.
    cevap = sor([{"role": "system", "content": rag_sistem("\n\n".join(bulunan))},   # System,
                 {"role": "user", "content": soru}], 160)                            # soru.
    return cevap, [p.split("\n")[0] for p in bulunan]   # Cevap + başlıklar.


def cevapla(mesaj, gecmis, mod):                  # Gradio'nun çağırdığı fonksiyon.
    """Moda göre sohbet / RAG / ajan."""
    try:                                          # Hata arayüzü çökertmesin.
        if mod == "Sohbet":                       # Düz sohbet:
            return sor([{"role": "user", "content": mesaj}], 120)   #   tek mesaj.
        if mod == "Rehber (RAG)":                 # RAG:
            cevap, kaynaklar = asistan_rag(mesaj) #   cevap + kaynaklar,
            return f"{cevap}\n\n_Kaynak: {', '.join(kaynaklar)}_"   #   kaynakları altına yaz.
        cevap, izler = araclar.ajan(mesaj, sor)   # Ajan: cevap + araç izleri,
        return cevap + (f"\n\n_🔧 {' · '.join(izler)}_" if izler else "")   #   izleri altına yaz.
    except Exception as hata:                     # Beklenmedik hata:
        return f"Bir sorun oldu: {type(hata).__name__}. Tekrar dener misiniz? {ISARET}"   # Kibar mesaj.


with open(TEST, encoding="utf-8") as f:           # Örnek sorular:
    ORNEKLER = [[r["soru"]] for r in list(csv.DictReader(f))[:3]]   #   paketin ilk 3 test sorusu.
demo = gr.ChatInterface(                          # Arayüz (defterdekiyle aynı):
    cevapla,                                      #   cevaplayan fonksiyon,
    additional_inputs=[gr.Radio(["Sohbet", "Rehber (RAG)", "Ajan"],   #   mod seçimi,
                                value="Rehber (RAG)", label="Mod")],  #   varsayılan RAG,
    title=f"{PROJE_ADI} {ISARET}",                #   başlık,
    description=f"{KURUM} asistanı · CPU'da çalışır, cevaplar 10-60 sn sürebilir",   # açıklama,
    examples=ORNEKLER,                            #   örnek sorular,
    cache_examples=False)                         #   açılışta modelle çalıştırma (CPU'da çöker).
if __name__ == "__main__":                        # Space bu dosyayı doğrudan çalıştırır:
    demo.launch()                                 #   sunucuyu başlat.

Writing app.py


### 10.3 Site klasörü

**Bu hücrede:** Siteye gidecek dosyaları `space/` klasöründe topluyoruz: `app.py` (yer tutucular deponuzla değiştirilir), `proje.py`, `araclar.py`, veri dosyaları, `requirements.txt` (llama-cpp-python için hazır CPU paketi adresi; sunucuda derleme beklenmez) ve Space'in `README.md` üst bilgisi.

In [25]:
import shutil                                     # Dosya kopyalama.
from pathlib import Path                          # Yol işlemleri.

SPACE = Path("space")                             # Siteye yüklenecek klasör.
SPACE.mkdir(exist_ok=True)                        # Yoksa oluştur.
app = Path("app.py").read_text(encoding="utf-8")  # Uygulama metni,
app = app.replace("GGUF_DEPO_YER", GGUF_DEPO).replace("GGUF_DOSYA_YER", GGUF_DOSYA)   # yer tutucular.
(SPACE / "app.py").write_text(app, encoding="utf-8")   # Siteye yaz.
for d in ["proje.py", "araclar.py", REHBER, TABLO, TEST]:   # Ortak kod ve veri dosyaları:
    shutil.copy(d, SPACE / d)                     #   kopyala.
WHEEL = "https://abetlen.github.io/llama-cpp-python/whl/cpu"   # llama-cpp-python hazır CPU paketleri.
(SPACE / "requirements.txt").write_text(          # Paket listesi:
    f"--extra-index-url {WHEEL}\n"                #   derleme yerine hazır paket,
    "llama-cpp-python\nhuggingface_hub\npypdf\nscikit-learn\nrequests\n", encoding="utf-8")
# Space kartı: üst bilgi (SDK, sürüm, giriş dosyası) + kısa açıklama.
(SPACE / "README.md").write_text(f"""---
title: {PROJE_ADI}
emoji: 💬
colorFrom: indigo
colorTo: green
sdk: gradio
sdk_version: {gr.__version__}
app_file: app.py
pinned: false
---
# {PROJE_ADI}
{KURUM} için ince ayarlı asistan. Model: [{GGUF_DEPO}](https://huggingface.co/{GGUF_DEPO}).
Modlar: Sohbet · Rehber (RAG, TF-IDF) · Ajan. CPU'da çalışır; cevaplar 10-60 sn sürebilir.
""", encoding="utf-8")                            # Üst bilgi + açıklama.
print(sorted(p.name for p in SPACE.iterdir()))    # Klasör içeriği.

['README.md', 'app.py', 'araclar.py', 'proje.py', 'rehber.pdf', 'requirements.txt', 'test_sorulari.csv', 'urunler.csv']


### 10.4 (İsteğe bağlı) Siteyi Colab'da denemek

**Bu hücrede:** Siteye göndermeden önce aynı uygulamayı burada, CPU'da çalıştırıyoruz: `llama-cpp-python` kurulur (~1 dk), GGUF Hub'dan iner, üç moda birer soru sorulur. Sitede de benzer hız beklenir. Zaman kısıtlıysa atlayabilirsiniz.

In [26]:
import sys                                        # İçe aktarma yolu.
!pip install -q llama-cpp-python --extra-index-url {WHEEL} 2>&1 | tail -1
sys.path.insert(0, str(SPACE))                    # space/ klasörünü içe aktarma yoluna ekle.
uygulama = importlib.import_module("app")         # app.py çalışır: GGUF iner, model yüklenir.
t0 = time.time()                                  # CPU hızını ölçelim.
print(uygulama.cevapla("Sen kimsin?", [], "Sohbet"))                          # Sohbet.
print(uygulama.cevapla(TEST_SORULARI[0]["soru"], [], "Rehber (RAG)"))         # RAG.
print(uygulama.cevapla(ajan_sorulari[0], [], "Ajan"))                         # Ajan.
print(f"3 cevap {time.time() - t0:.0f} sn (CPU)")   # Sitede benzer hız beklenir.
araclar.ARA = chroma_ara                          # Defterdeki ajan yeniden Chroma'yı kullansın.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.1 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


./kivi-1.5b-q8_0.gguf: reconstructing file:   0%|          |  0.00B / 1.65GB            

./kivi-1.5b-q8_0.gguf: downloading bytes:           |  0.00B            

Ben Kivi, Kivi Elektronik'in müşteri hizmetleri asistanıyım. Size nasıl yardımcı olabilirim?
Kargo ücreti 79 TL'dir. 🥝

_Kaynak: 3. Kargo, 11. Ön sipariş_
Pixa 65 inç OLED Televizyon'un fiyatı 64999 TL ve stokta yoktur. 🥝

_🔧 urun_bilgisi(Pixa 65 inç OLED Televizyon)_
3 cevap 102 sn (CPU)


### 10.5 Yayın

**Bu hücrede:** Önce Space açmayı deniyoruz. Hesabınız PRO ise site birkaç dakika içinde "Building" → "Running" olur. Değilse Hugging Face `402 Payment Required` döner; o zaman `space/` klasörünüz GGUF deponuza yüklenir ve **eğitmen sitenizi yayınlar**. Her iki durumda da teslim edeceğiniz linkler hücrenin çıktısında yazar.

In [27]:
SPACE_DEPO = f"{KULLANICI}/{MODEL_ADI.lower()}"   # Space adı (adresin son parçası).
atla = ["__pycache__/*", "*.pyc"]                 # Önbellek dosyaları gitmesin.
try:                                              # Önce kendi Space'inizi deneyin:
    api.create_repo(SPACE_DEPO, repo_type="space",   #   Space deposu aç,
                    space_sdk="gradio", exist_ok=True, private=False)   #   Gradio, herkese açık,
    api.upload_folder(folder_path="space", repo_id=SPACE_DEPO,   #   space/ klasörünü yükle,
                      repo_type="space", ignore_patterns=atla)  #   önbelleksiz.
    SITE = f"https://huggingface.co/spaces/{SPACE_DEPO}"   # Site adresi.
    print("Site:", SITE, "(3-5 dk sonra 'Running'; sorun olursa 'Logs' sekmesi)")   # Bilgi.
except Exception as hata:                         # Hata:
    if "402" not in str(hata):                    #   402 değilse gerçek bir sorun var:
        raise                                     #   hatayı göster.
    api.upload_folder(folder_path="space", path_in_repo="space", repo_id=GGUF_DEPO,   # Space dosyaları
                      ignore_patterns=atla)       #   GGUF deposunun space/ klasörüne.
    SITE = "eğitmen vitrininde yayınlanacak"      # Model kartında yazacak.
    print("Hesabınız Space açamıyor (402: PRO gerekiyor). Site dosyaları:")   # Durum,
    print(f"  https://huggingface.co/{GGUF_DEPO}/tree/main/space")          # dosyaların yeri,
    print("Teslimde GGUF deposu linkini verin; eğitmen sitenizi yayınlayacak.")   # ne yapılacak.

Hesabınız Space açamıyor (402: PRO gerekiyor). Site dosyaları:
  https://huggingface.co/sedaahi/Kivi-1.5B-GGUF/tree/main/space
Teslimde GGUF deposu linkini verin; eğitmen sitenizi yayınlayacak.


**Sorun çıkarsa:** (1) *RUNTIME_ERROR / 502* → `cache_examples=False` duruyor mu? Sayfadaki "Logs" sekmesinde hata satırına bakın. (2) *Build hatası* → Logs'ta hangi paket? `requirements.txt`'yi düzeltip yükleme hücresini yeniden çalıştırın. (3) *Model inmiyor* → GGUF deposu herkese açık mı? (4) *Çok yavaş* → 3B model CPU'da 1.5B'den yaklaşık iki kat yavaştır; cevap uzunluğunu (`maks`) azaltın.

## 11. Model Kartı = Proje Raporu

**Bu hücrede:** Modelinizin Hub sayfasındaki README'yi (model kartı) yazıyoruz. Bu kart **proje raporunuzdur**: ne yaptınız, neden bu taban modeli seçtiniz, veri nasıl üretildi, önce/sonra ölçümleri, test soruları sonuçları, gözlemler ve sınırlar. Sayılar ölçümlerden gelir; **metin kısımlarını siz yazın** (hücrenin başındaki dört değişken).

In [31]:
import os
import json
from huggingface_hub import hf_hub_download

# ===== PROJE RAPORU: DÖRT PARAGRAF =====

# Proje ne yapıyor?
PROJE_OZETI = (
    "Kivi, kurgusal Kivi Elektronik mağazası için Seda Ahi tarafından "
    "geliştirilen Türkçe müşteri hizmetleri asistanıdır. Ürün bilgileri, "
    "sipariş, ödeme, kargo, iade, değişim ve garanti konularında yardımcı "
    "olmayı amaçlar. Kimliği, üslubu ve araç kullanma davranışı QLoRA "
    "ince ayarıyla öğretilmiştir. Kurum kuralları rehber.pdf üzerinden "
    "RAG ile, ürünlerin fiyat, stok ve garanti bilgileri urunler.csv "
    "üzerinden araçla alınır. Rehber arama, ürün arama ve hesaplama "
    "araçlarına ek olarak sepet tutarına göre çalışan kargo_ucreti "
    "aracı geliştirilmiştir. Arayüz Sohbet, Rehber (RAG) ve Ajan "
    "olmak üzere üç mod sunar."
)

# Neden bu taban model?
TABAN_GEREKCESI = (
    "Qwen/Qwen2.5-1.5B-Instruct; hazır sohbet şablonu, Türkçe "
    "kullanımına uygunluğu, Apache-2.0 lisansı ve sınırlı GPU "
    "kaynaklarıyla ince ayar yapılabilmesi nedeniyle seçilmiştir. "
    "Küçük model boyutu, Colab T4 ortamında eğitim ve GGUF biçiminde "
    "CPU üzerinde çalıştırma imkanı sağlamıştır. Bu çalışmada eğitim "
    "16,7 dakika sürmüş ve tepe GPU bellek kullanımı 3,56 GB olmuştur. "
    "GGUF CPU denemesinde üç cevap toplam 102 saniyede üretilmiştir. "
    "Bu çalışmada başka bir taban model denenmemiştir."
)

# Eğitim verisinde, ölçümlerde ve denemelerde ne görüldü?
GOZLEMLER = (
    "Gemini ile 150 kimlik, 167 RAG, 120 tek adımlı ajan kararı ve "
    "38 iki turlu ajan örneği olmak üzere toplam 475 sentetik örnek "
    "üretilmiş; 428 örnek eğitim, 47 örnek değerlendirme için ayrılmıştır. "
    "Gözle incelenen örneklerde genel olarak kimlik ve işaret korunmuş, "
    "ancak bazı cevaplarda kaynak anlamının değişebildiği görülmüştür. "
    "Örneğin para iadesinin 3 iş günü içinde başlatılması kuralı, bir "
    "örnekte 3 iş günü sonra başlatılması şeklinde ifade edilmiştir. "
    "Eğitim örneklerinin tamamı 1024 token sınırının altında kalmıştır. "
    "Üç epoch boyunca doğrulama loss değerleri 0,461, 0,223 ve 0,207 "
    "olarak düşmüştür. Eğitim sonrası kimlikte ad ve işaret oranları "
    "0'dan 1'e, geçerli JSON oranı 0,81'den 1'e, doğru araç seçimi "
    "0,12'den 0,94'e yükselmiştir. RAG hakem puanı ise 0,47'den "
    "0,53'e sınırlı artış göstermiştir. Bu ölçümler tam cevap "
    "doğruluğunu garanti etmemiş; ayrı 15 soruluk uçtan uca testte "
    "ortalama puan 0,37 olmuştur. Bağlamsız cevaplar kuralları "
    "uydurabilirken, RAG iade süresi gibi bazı soruları düzeltmiştir. "
    "Bununla birlikte doğru bölüm getirildiğinde bile kargo koşullarını "
    "atlama, hijyen istisnasını yanlış yorumlama ve kaynakta bulunmayan "
    "bilgiler üretme görülmüştür. Tek bölümle arama ve ek cevap "
    "talimatları ayrı deneyler olarak denenmiş, tutarlı iyileşme "
    "sağlamadığından asıl RAG akışı değiştirilmemiştir. Ajan ürün "
    "fiyatı ve stok bilgisini doğru aktarabilmiş, ancak ürün aramasından "
    "sonra hesaplama gerektiren soruda ikinci aracı çağırmamıştır. "
    "750 TL sepet sorusunda kargo aracını doğru kullanırken, tam "
    "1000 TL sorusunda yanlış araç seçmiştir. Gradio'nun üç modu "
    "ve GGUF tabanlı CPU uygulaması denenmiştir. "
    "'Kargo ücreti ne kadar?' sorusunda Chroma araması "
    "'3. Kargo' ve '2. Ödeme seçenekleri' bölümlerini getirirken, "
    "CPU uygulamasındaki TF-IDF araması '3. Kargo' ve "
    "'11. Ön sipariş' bölümlerini getirmiştir. Her iki yöntemde de "
    "cevap 79 TL bilgisini vermiş ancak ücretsiz kargo koşulunu "
    "atlamıştır. Bu örnekte ikinci kaynak değişmiş, doğru kargo "
    "bölümü bulunmasına rağmen cevap eksik kalmıştır."
)

# Asistan ne yapamaz? Kullanıcı neye dikkat etmeli?
SINIRLAR = (
    "Bu çalışma eğitim amaçlı bir prototiptir; gerçek bir mağazanın "
    "resmi asistanı değildir. Gerçek müşteri hesaplarına, siparişlere "
    "ve canlı stok sistemlerine erişmez. Kurum bilgileri yalnızca "
    "sağlanan kurgusal rehber ve ürün tablosuyla sınırlıdır. Küçük "
    "model, bağlam verilse bile koşulları atlayabilir, çelişkili "
    "evet/hayır cevapları verebilir ve rehberde bulunmayan bilgileri "
    "uydurabilir. Kimlikte ad kullanma ve JSON biçimi ölçümleri "
    "anlamsal doğrulukla aynı değildir. Çok adımlı araç kullanımı "
    "ve sınır değer içeren sorularda güvenilirliği düşüktür. "
    "Gemini hakem puanları otomatik değerlendirmedir; insan "
    "incelemesinin yerini tutmaz. Colab'da Chroma ve çok dilli "
    "embedding, CPU site uygulamasında ise TF-IDF araması "
    "kullanıldığından bulunan kaynaklar ve cevaplar farklılaşabilir. "
    "CPU cevapları yavaştır. Space oluşturma isteği 402 hatasıyla "
    "reddedildiği için site dosyaları GGUF deposunun space/ "
    "klasörüne yüklenmiştir; kalıcı yayın eğitmen vitrini "
    "üzerinden yapılmayı beklemektedir."
)

LISANS = "apache-2.0"


# ===== ÖLÇÜMLERİ OKU =====

olcum_yolu = (
    "olcumler.json"
    if os.path.exists("olcumler.json")
    else hf_hub_download(
        repo_id=VERI_SETI,
        filename="olcumler.json",
        repo_type="dataset",
    )
)

with open(olcum_yolu, encoding="utf-8") as f:
    olc = json.load(f)

ONCE = olc["once"]
SONRA = olc["sonra"]


# ===== TEST SONUÇLARINI OKU =====

if "SONUCLAR" not in globals():
    test_yolu = hf_hub_download(
        repo_id=VERI_SETI,
        filename="test_sonuclari.json",
        repo_type="dataset",
    )

    with open(test_yolu, encoding="utf-8") as f:
        SONUCLAR = json.load(f)

if not SONUCLAR:
    raise ValueError("Test sonuçları boş; model kartı oluşturulmadı.")

TEST_PUANI = (
    sum(float(s.get("puan", 0)) for s in SONUCLAR)
    / len(SONUCLAR)
)


# ===== MARKDOWN TABLOLARINI HAZIRLA =====

def tablo_metni(deger):
    """Tablo hücrelerindeki satır sonlarını ve dikey çizgileri düzenler."""
    return (
        str(deger)
        .replace("|", r"\|")
        .replace("\r", " ")
        .replace("\n", " ")
    )


satirlar = "\n".join(
    f"| {tablo_metni(k)} | {ONCE[k]:.2f} | {SONRA[k]:.2f} |"
    for k in ONCE
    if k != "sure_sn"
)

# Cevaplar kesilmeden tabloya eklenir.
test_satirlari = "\n".join(
    f"| {tablo_metni(s['soru'])} "
    f"| {tablo_metni(s['cevap'])} "
    f"| {s.get('puan', '-')} |"
    for s in SONUCLAR
)

site_dosyalari = (
    f"https://huggingface.co/{GGUF_DEPO}/tree/main/space"
)


# ===== MODEL KARTINI OLUŞTUR =====

KART = f"""---
language: [tr]
license: {LISANS}
base_model: {TABAN}
tags: [qlora, turkish, rag, agent, bootcamp]
datasets: [{VERI_SETI}]
---
# {MODEL_ADI} {ISARET}

{PROJE_OZETI}

**Bağlantılar:** [LoRA adaptörü](https://huggingface.co/{ADAPTOR}) ·
[GGUF](https://huggingface.co/{GGUF_DEPO}) ·
[Eğitim verisi](https://huggingface.co/datasets/{VERI_SETI}) ·
[Site dosyaları]({site_dosyalari})

**Yayın durumu:** Space oluşturma isteği 402 hatasıyla reddedilmiştir.
Site dosyaları GGUF deposunun `space/` klasörüne yüklenmiş olup
eğitmen vitrini üzerinden yayınlanmayı beklemektedir.

## Taban model

`{TABAN}`. {TABAN_GEREKCESI}

## Veri ve eğitim

Toplam 475 sentetik sohbet örneği Gemini ile üretilmiştir:
150 kimlik, 167 RAG, 120 tek adımlı ajan kararı ve 38 iki turlu ajan örneği.

428 örnek eğitim, 47 örnek değerlendirme için ayrılmıştır.
Ayrıca veri paketindeki 13 soruya, geliştirilen kargo aracını gerektiren
2 soru eklenerek 15 soruluk uçtan uca değerlendirme yapılmıştır.

QLoRA: r=16, alpha=32, tüm doğrusal katmanlar, 3 epoch,
öğrenme hızı 2e-4, maksimum örnek uzunluğu 1024 token.

## Ölçümler (önce → sonra, değerlendirme seti)

| Ölçüt | Taban | {PROJE_ADI} |
|---|---|---|
{satirlar}

## Test soruları

Gemini hakem ölçeği: 1 / 0.5 / 0.
Soru sayısı: {len(SONUCLAR)}
Ortalama puan: **{TEST_PUANI:.2f} / 1**

| Soru | Cevap | Puan |
|---|---|---|
{test_satirlari}

## Gözlemler

{GOZLEMLER}

## Sınırlar

{SINIRLAR}
"""


# ===== HUGGING FACE MODEL KARTINI GÜNCELLE =====

api.upload_file(
    path_or_fileobj=KART.encode("utf-8"),
    path_in_repo="README.md",
    repo_id=TAM_MODEL,
    repo_type="model",
)

print("Model kartı başarıyla güncellendi.")
print("Test sorusu sayısı:", len(SONUCLAR))
print(f"Ortalama test puanı: {TEST_PUANI:.2f}")
print("Model kartı:", f"https://huggingface.co/{TAM_MODEL}")
print("Site dosyaları:", site_dosyalari)

Model kartı başarıyla güncellendi.
Test sorusu sayısı: 15
Ortalama test puanı: 0.37
Model kartı: https://huggingface.co/sedaahi/Kivi-1.5B
Site dosyaları: https://huggingface.co/sedaahi/Kivi-1.5B-GGUF/tree/main/space


## Teslim

Teslim formuna şu linkleri girin:

| Ne | Nerede |
|---|---|
| Colab defteriniz | `Paylaş → Bağlantıya sahip olan herkes: Görüntüleyen` |
| Model (kart = rapor) | `https://huggingface.co/kullanıcı-adınız/MODEL_ADI` |
| GGUF deposu | `https://huggingface.co/kullanıcı-adınız/MODEL_ADI-GGUF` |
| Eğitim verisi | `https://huggingface.co/datasets/kullanıcı-adınız/model_adı-veri` |
| Site | Kendi Space'iniz ya da "eğitmen vitrini" (GGUF deposunda `space/` klasörü) |

**Son kontrol:** Defter baştan sona hatasız çalışıyor mu? Model kartında dört paragraf sizin mi? Proje kartındaki ad, kimlik ve dosyalar sizin paketinize ait mi? Kendi aracınız eğitim verisinde ve sitede çalışıyor mu?

## Kivi Projesi — Teslim Bağlantıları

| İstenen çıktı | Bağlantı |
|---|---|
| Colab defteri | [Kivi çalışma defteri](https://colab.research.google.com/drive/1qj3FB3zO5_Y8rhPxbqb9qtfJqn8_t9XF?usp=sharing) |
| Model ve proje raporu | [Kivi-1.5B](https://huggingface.co/sedaahi/Kivi-1.5B) |
| LoRA adaptörü | [Kivi-1.5B-LoRA](https://huggingface.co/sedaahi/Kivi-1.5B-LoRA) |
| GGUF deposu | [Kivi-1.5B-GGUF](https://huggingface.co/sedaahi/Kivi-1.5B-GGUF) |
| Eğitim verileri ve ölçümler | [kivi-1.5b-veri](https://huggingface.co/datasets/sedaahi/kivi-1.5b-veri) |
| Site | Eğitmen vitrini — yayın bekleniyor |
| Yayına hazır site dosyaları | [GGUF deposundaki space klasörü](https://huggingface.co/sedaahi/Kivi-1.5B-GGUF/tree/main/space) |

Space oluşturma isteği 402 hatasıyla reddedildiği için yönergede
belirtilen alternatif yol kullanılmıştır. Site dosyaları eğitmenin
yayınlayabilmesi için yüklenmiştir. Gradio demosu ve GGUF CPU
uygulaması Colab'da denenmiştir.